In [1]:
# =============================================================================
#  EXP27 FOLLOW-UPS — SELF-CONTAINED. Rebuilds everything from scratch; needs no live kernel.
#
#  WHY THIS FILE EXISTS. EXP27c (logit lens) and EXP27d (bin-only oracle) are already run and are
#  not repeated here. EXP27d's oracle_first = 1.000 is VOID and must not be reported: it trained
#  on the 570-item bin, and 570 < 3584 (the donor hidden dim) means those donor states are
#  linearly independent, so a linear map can place each one independently and a perfect score was
#  guaranteed regardless of donor content. EXP27g below replaces it properly.
#
#  WHAT IT RUNS
#    RUN CONFIG -> CELL D (build facts) -> EXP27b core (bin, recon, ONE task-map seed)
#    -> EXP27e class sweep, now with a per-K SHUFFLED control
#    -> EXP27f class-matched training (arms A / B / C)
#    -> EXP27g pooled oracle (train+eval, 27k items, comparable to arithmetic EXP9)
#    -> save to exp27_followups_results.json
#
#  ONE SEED, NOT FIVE. The 5-seed task map is already measured (0.351 [0.344, 0.357]); here the
#  task map is only a comparison baseline for 27f/27g, so FACT_SEEDS = [0]. That cuts the largest
#  cost by 4/5. Everything upstream is seeded (split via random.Random(0), greedy decoding), so
#  the bin should reproduce at n = 570 -- if it does not, something upstream drifted and the
#  EXP27b core block will show it before anything expensive runs.
#
#  RUN ORDER: on a fresh pod run cells 1-4 then RESTART KERNEL. Otherwise start at cell 5.
#  Paste your HF token in the login cell either way.
# =============================================================================
print("EXP27 follow-ups — self-contained. Run RUN CONFIG before CELL D.")


EXP27 follow-ups — self-contained. Run RUN CONFIG before CELL D.


In [2]:
# === CELL 1: Dependencies — run cells 0-3 once, in order, then RESTART KERNEL ===
# One resolved install so pip solves versions a SINGLE time, before any model is loaded:
#   * transformers is pinned to the version the model code targets (4.46.3)
#   * sae_lens (for EXP16 / EXP20 / EXP21) is installed HERE, not mid-run, so it can't retug
#     torch/transformers after the models are already sitting in memory.
# The cu128 torch fix in the NEXT cell runs AFTER this line, so on Blackwell / sm_120 pods it
# always wins over whatever torch sae_lens's resolver pulls. (If pip reports a hard transformers
# conflict from sae_lens, drop sae_lens from this line, `pip install sae_lens` on its own, then
# re-run this pinned line so transformers is restored to 4.46.3.)
%pip install -q "transformers==4.46.3" accelerate bitsandbytes numpy matplotlib datasets hf_transfer sae_lens


Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install --upgrade torch --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip uninstall -y torchvision torchaudio
%pip install -U --force-reinstall "typing_extensions>=4.12"


Note: you may need to restart the kernel to use updated packages.
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.16.0
    Uninstalling typing_extensions-4.16.0:
      Successfully uninstalled typing_extensions-4.16.0
Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
import torch
import transformers
import sae_lens

print("Python:", sys.executable)
print("Torch location:", torch.__file__)
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("SAE Lens:", sae_lens.__version__)

Python: /venv/main/bin/python
Torch location: /venv/main/lib/python3.12/site-packages/torch/__init__.py
Torch: 2.14.0+cu130
CUDA: 13.0
GPU: NVIDIA L40S
Transformers: 4.46.3
SAE Lens: 6.5.3


In [2]:
# verify the resolved stack BEFORE restarting (catch a bad resolve early, not 40 min into a run)
# NOTE: typing_extensions has no __version__ attribute, so do not print one. torch importing at
# all IS the check -- a stale typing_extensions makes `import torch` raise on `TypeIs`.
import torch
print("torch:", torch.__version__, "| cuda cap:", torch.cuda.get_device_capability(0))
import transformers, sae_lens
print("transformers:", transformers.__version__, "(want 4.46.3)  | sae_lens:", sae_lens.__version__)


torch: 2.14.0+cu130 | cuda cap: (8, 9)
transformers: 4.46.3 (want 4.46.3)  | sae_lens: 6.5.3


In [3]:
# === CELL 2:imports, set_submodule shim, global config (VRAM/compute switches, primarily smoke test) ===
import os, json, math, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# shim: newer transformers' 4-bit path calls nn.Module.set_submodule, absent on older torch
if not hasattr(nn.Module, "set_submodule"):
    def _set_submodule(self, target, module):
        mod = self
        atoms = target.split(".")
        for a in atoms[:-1]:
            mod = getattr(mod, a)
        setattr(mod, atoms[-1], module)
    nn.Module.set_submodule = _set_submodule

DEVICE = "cuda"
torch.manual_seed(0)
MODEL_2B = "google/gemma-2-2b"
MODEL_9B = "google/gemma-2-9b"

# Smoke test lever, (True for 5 min test eval False for full eval)
SMOKE_TEST = False            

# validated layers (Different for each model pair)
SINGLE_PAIR = (20, 34)
LAYER_PAIRS = [(18, 31), (20, 34), (22, 37), (24, 40)]
L2_SINGLE, L9_SINGLE = SINGLE_PAIR
PATCH_POS = -1
RIDGE_LAMBDA = 1e3

if SMOKE_TEST:
    N_PATCH, N_ARITH_TRAIN, N_ARITH_EVAL = 12, 200, 120
    GSM_FIT, GSM_EVAL, GSM_STRENGTHS = 20, 24, [1.0]
    TASK_SEEDS, TASK_EPOCHS = [0, 1], 2
    BOOT_B = 1000
else:
    N_PATCH, N_ARITH_TRAIN, N_ARITH_EVAL = 300, 3000, 2000      # ~720 unsolvable muladd
    GSM_FIT, GSM_EVAL, GSM_STRENGTHS = 150, 1319, [1.0]    # full GSM8K test set
    TASK_SEEDS, TASK_EPOCHS = [0, 1, 2, 3, 4], 6                # 5 seeds for the task map
    BOOT_B = 10000

ARITH_BATCH, GSM_BATCH, MAX_NEW_GSM, MAX_NEW_ARITH = 16, 8, 300, 8
RESULTS = {}   # everything defensible gets concentrated here and printed at the end
print("SMOKE_TEST =", SMOKE_TEST)

SMOKE_TEST = False


In [ ]:
# === CELL 3: Hugging Face login (Gemma is gated) ===
from huggingface_hub import login
login("")

In [5]:
# === CELL 4: statistics helpers (match the interval to the source of randomness) ===
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),)*3
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = (z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)))/d
    return p, max(0.0, c-h), min(1.0, c+h)
def wilson_bools(b, z=1.96):
    b = np.asarray(b, bool); return wilson(int(b.sum()), int(b.size), z)
def bootstrap_ci(x, B=None, alpha=0.05, seed=0):
    x = np.asarray(x, float); n = len(x); B = B or BOOT_B
    if n == 0: return (float("nan"),)*3
    rng = np.random.default_rng(seed)
    m = x[rng.integers(0, n, (B, n))].mean(1)
    lo, hi = np.percentile(m, [100*alpha/2, 100*(1-alpha/2)])
    return float(x.mean()), float(lo), float(hi)
_TC = {2:12.706,3:4.303,4:3.182,5:2.776,6:2.571,7:2.447,8:2.365,9:2.306,10:2.262}
def across_seed_ci(v, alpha=0.05):
    v = np.asarray(v, float); k = len(v); m = float(v.mean())
    if k < 2: return (m, float("nan"), float("nan"))
    se = v.std(ddof=1)/math.sqrt(k); t = _TC.get(k, 1.96)
    return (m, m-t*se, m+t*se)
def fmt(tr): p, lo, hi = tr; return f"{p:.3f} [{lo:.3f}, {hi:.3f}]"

In [6]:
# === CELL 5: shared helpers (hooks, padding, ridge map, problems, GSM8K, full-answer) ===
def _hid(o): return o[0] if isinstance(o, tuple) else o
def _pack(o, h): return (h,)+tuple(o[1:]) if isinstance(o, tuple) else h
def capture(store, key):
    def hook(_m,_i,o): store[key] = _hid(o)[:, PATCH_POS, :].detach()
    return hook
def patch_vec(vec):  # replace last-pos with vec (graph-safe: works under autograd too)
    def hook(_m,_i,o):
        h = _hid(o)
        if h.shape[1] == 0: return o
        h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
        return _pack(o, h2)
    return hook
 
def left_pad(id_list, pad_id):
    L = max(t.numel() for t in id_list)
    ids = torch.full((len(id_list), L), pad_id, dtype=torch.long)
    m = torch.zeros((len(id_list), L), dtype=torch.long)
    for i, t in enumerate(id_list):
        t = t.flatten(); ids[i, L-t.numel():] = t; m[i, L-t.numel():] = 1
    return ids, m
 
def fit_ridge(X9, X2, lam=RIDGE_LAMBDA):
    mu9, mu2 = X9.mean(0), X2.mean(0)
    A, B = X9-mu9, X2-mu2
    W = torch.linalg.solve(A.T@A + lam*torch.eye(A.shape[1]), A.T@B)
    return mu9, mu2, W
def apply_map(x, m): mu9, mu2, W = m; return (x-mu9)@W + mu2
 
# ---- arithmetic problems (muladd only: healthy unsolvable bin) ----
FEWSHOT = ("2 + 5 = 7\n6 * 3 = 18\n4 * 7 + 2 = 30\n9 * 8 = 72\n"
           "3 * 4 + 5 = 17\n40 * 20 = 800\n")
def _aprompt(expr): return f"{FEWSHOT}{expr} ="
def _aencode(tok, expr, ans):
    # Build prompt and prompt+answer, then target the FIRST answer token that carries a
    # digit. On Gemma the answer tokenizes as [space, digit] so that token is at len(p)+1;
    # on byte-level BPE tokenizers (Qwen, Llama) the leading space fuses with the first
    # digit, so it's at len(p). Scanning for the first digit-bearing token handles both,
    # plus any tokenizer that emits leading whitespace/markup tokens before the number.
    p = tok(_aprompt(expr)).input_ids
    f = tok(_aprompt(expr) + " " + str(ans)).input_ids
    if f[:len(p)] != p or len(f) <= len(p): return None, None
    j = len(p)
    while j < len(f) and not any(ch.isdigit() for ch in tok.decode([f[j]])): j += 1
    if j >= len(f): return None, None
    return torch.tensor(f[:j]), f[j]   # context up to (not incl.) the first digit token; target = that token
def gen_arith(tok, n, rng, exclude=None):
    exclude = exclude or set(); out, seen = [], set()
    tries = 0
    while len(out) < n and tries < n*120:
        tries += 1
        a, b, c = rng.randint(2, 40), rng.randint(2, 40), rng.randint(1, 99)
        expr, ans = f"{a} * {b} + {c}", a*b+c
        if expr in seen or expr in exclude: continue
        seen.add(expr)
        ids, tok_id = _aencode(tok, expr, ans)
        if tok_id is None: continue
        out.append(dict(expr=expr, ans=ans, ids=ids, tok=tok_id))
    return out
 
# ---- GSM8K: validated prompt + extraction (from the Linear_gsm8k notebook) ----
import re as _re
def gsm_prompt(q):
    return (
        "Below are math problems with detailed step-by-step solutions.\n\n"
        "Problem: Natalia sold clips to 48 of her friends in April, and then she sold "
        "half as many clips in May. How many clips did Natalia sell altogether in April and May?\n"
        "Solution: Let's think step-by-step.\n"
        "1. Clips sold in April: 48\n"
        "2. Clips sold in May: 48 / 2 = 24\n"
        "3. Total clips: 48 + 24 = 72\n"
        "#### 72\n\n"
        f"Problem: {q}\n"
        "Solution: Let's think step-by-step."
    )
def gsm_extract(text):
    m = _re.search(r"####\s*(-?[\d,.]+)", text)
    if m:
        try: return float(m.group(1).replace(",", ""))
        except ValueError: pass
    m = _re.search(r"answer is\s*(-?[\d,.]+)", text, _re.IGNORECASE)
    if m:
        try: return float(m.group(1).replace(",", ""))
        except ValueError: pass
    nums = _re.findall(r"-?[\d,.]+", text)
    if nums:
        try: return float(nums[-1].rstrip(".").replace(",", ""))
        except ValueError: return None
    return None
def _parse_first_int(text):
    """Arithmetic answers: take the FIRST integer the model emits after '=',
    not the last (the model may continue with few-shot-style lines)."""
    m = _re.search(r"-?\d+", text)
    return float(m.group()) if m else None
 
# ---- generic batched forward: last-pos resid at given layers + top token ----
@torch.inference_mode()
def states_and_top(model, layers, prob_ids, batch=ARITH_BATCH):
    base = model.model if hasattr(model, "model") else model
    acc = {L: [] for L in layers}; top = []
    for i in range(0, len(prob_ids), batch):
        ids, m = left_pad(prob_ids[i:i+batch], tokenizer.pad_token_id)
        out = model(ids.to(DEVICE), attention_mask=m.to(DEVICE), output_hidden_states=True)
        for L in layers: acc[L].append(out.hidden_states[L+1][:, -1, :].float().cpu())
        top += out.logits[:, -1, :].argmax(-1).cpu().tolist()
    return {L: torch.cat(v) for L, v in acc.items()}, top
 
# ---- per-batch graft hook: replace last prompt-position resid with vec[B,d] ----
# Fires only during prefill (seq len > 1); no-ops during generation (len==1) and
# when the batch dim doesn't match, so generation proceeds normally after seeding.
_graft = {"vec": None}
def patch_vec_batch(_m, _i, o):
    h = _hid(o); vec = _graft["vec"]
    if vec is None or h.shape[1] <= 1 or h.shape[0] != vec.shape[0]:
        return o
    h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
    return _pack(o, h2)
 
@torch.inference_mode()
def arith_fullanswer_correct(model, layer, probs, vecs=None, batch=ARITH_BATCH):
    """Generate the full number and compare to gold. If vecs is given, vecs[i] is
    grafted at the last prompt position of problem i (prefill) before generation.
    Returns list[bool], one per problem."""
    ok, handle = [], None
    if vecs is not None:
        handle = model.model.layers[layer].register_forward_hook(patch_vec_batch)
    try:
        for i in range(0, len(probs), batch):
            chunk = probs[i:i+batch]
            ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
            ids, m = ids.to(DEVICE), m.to(DEVICE)
            _graft["vec"] = (torch.stack(vecs[i:i+len(chunk)]).to(DEVICE)
                             if vecs is not None else None)
            gen = model.generate(ids, attention_mask=m, max_new_tokens=MAX_NEW_ARITH,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
            txt = tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
            for p, t in zip(chunk, txt):
                pred = _parse_first_int(t)
                ok.append(pred is not None and abs(pred - p["ans"]) < 0.5)
    finally:
        if handle: handle.remove()
        _graft["vec"] = None
    return ok
 
print("helpers defined")

helpers defined


In [7]:
# === CELL 6: load both models once; donor in bf16 by default (4-bit optional) ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_2B)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
 
# QUANTIZE_9B: set True only when the donor won't fit in bf16 (e.g. the original Gemma-2-9B on
# a small card). For 7-8B donors (Qwen-7B ~15GB, Llama-8B ~16GB) on a 20GB+ card, keep this
# False — bf16 is correct and avoids a serious failure mode: under 4-bit, this transformers/bnb
# build runs unquantized layers in FP16, and Qwen/Llama activations OVERFLOW fp16 (>65504) ->
# NaN logits -> argmax collapses to token 0 ('!'). bf16 has the exponent range to avoid this.
QUANTIZE_9B = globals().get("QUANTIZE_9B", False)
 
model_2b = AutoModelForCausalLM.from_pretrained(
    MODEL_2B, torch_dtype=torch.bfloat16, attn_implementation="eager",
    low_cpu_mem_usage=True).to(DEVICE).eval()
if QUANTIZE_9B:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_use_double_quant=True)
    model_9b = AutoModelForCausalLM.from_pretrained(
        MODEL_9B, quantization_config=bnb, device_map={"": 0},
        torch_dtype=torch.bfloat16, attn_implementation="eager", low_cpu_mem_usage=True).eval()
else:
    model_9b = AutoModelForCausalLM.from_pretrained(
        MODEL_9B, torch_dtype=torch.bfloat16, attn_implementation="eager",
        low_cpu_mem_usage=True).to(DEVICE).eval()
print("loaded:", model_2b.config.num_hidden_layers, "x2B layers,",
      model_9b.config.num_hidden_layers, "x9B layers |",
      "9B quantized" if QUANTIZE_9B else "9B bf16")
 
# Health check: catch a NaN/overflow blowup (the fp16-under-4bit failure) immediately, not
# 168 silently-filtered pairs later. A healthy donor tops a real word here, never token 0.
with torch.inference_mode():
    _hl = model_9b(tokenizer("The capital of France is", return_tensors="pt").to(DEVICE).input_ids).logits[0, -1, :]
assert not torch.isnan(_hl).any() and not torch.isinf(_hl).any(), (
    "9B produced NaN/Inf logits — numerical blowup. If QUANTIZE_9B=True, the donor is overflowing "
    "fp16; set QUANTIZE_9B=False to load it in bf16 (needs the VRAM but is numerically safe).")
del _hl
 
# Same-family requirement: the two models must share the TOKENIZER so positions align
# (the whole stitch grafts by position). NOTE: config.vocab_size is the *padded embedding*
# count, not the tokenizer — Qwen pads differently across sizes (0.5B=151936, 7B=152064)
# while sharing one tokenizer, so comparing config.vocab_size gives false alarms. Verify the
# tokenizer itself instead, by checking a probe string maps to identical ids under each model's
# own tokenizer. (We load one shared tokenizer, but this also catches an accidental mismatch.)
_tk2 = AutoTokenizer.from_pretrained(MODEL_2B); _tk9 = AutoTokenizer.from_pretrained(MODEL_9B)
_probe = "3 * 12 + 7 = 43\nThe answer is 256."
assert _tk2(_probe).input_ids == _tk9(_probe).input_ids, (
    "tokenizer mismatch: the two models tokenize the same text differently, so positions won't "
    "align. This notebook requires a SAME-FAMILY pair sharing one tokenizer.")
del _tk2, _tk9

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

...

loaded: 26 x2B layers, 42 x9B layers | 9B bf16


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [8]:
# === RUN CONFIG — run this BEFORE CELL D and before EXP27b ===
# HARD ASSIGNMENTS, not globals().get() defaults. Your kernel from the previous run still holds
# FACT_MAX=8000, FACT_LR=1e-3, FACT_MAX_STEPS=2500, FACT_SEEDS=[0,1,2,3,4]; every globals().get()
# default further down would find those and silently keep the OLD value. This cell makes that
# impossible. (FACT_RANK / FACT_WD / FACT_MIN_CLASS_COUNT / FACT_SPLIT / FACT_MON_DENSE are new
# names, so they would pick up the new defaults anyway — they are listed here for one readout.)
FACT_SPLIT           = "train"   # TriviaQA rc.nocontext: ~138k rows; validation has only ~17.9k
FACT_MAX             = 40000     # candidates to encode (was 8000 -> 1.8 examples per class)
FACT_MIN_CLASS_COUNT = 2         # drop singleton answer classes from the CE LOOP ONLY
FACT_RANK            = 32        # low-rank residual on the recon init; 0 = old full-rank
FACT_WD              = 1e-2      # weight decay on A,B == anchoring toward the recon map
FACT_LR              = 1e-4      # was 1e-3: TRAIN conferral hit 0.94 by step 400
FACT_EPOCHS          = 4
FACT_MAX_STEPS       = 3000
FACT_MON_STEPS       = 100       # held-out check every N steps...
FACT_MON_DENSE       = 300       # ...but every 25 steps for the first 300
FACT_KEEP_BEST       = True
FACT_SEEDS           = [0]       # baseline only; the 5-seed number is already measured
RUN_CONFIG_APPLIED   = "v3"      # sentinel: EXP27b checks for this and refuses to run without it

# Which downstream experiments run. Each is a separate cell below and each reuses the state
# EXP27b leaves in the kernel -- none of them retrain the main map or rebuild the bin.
RUN_CLASSSWEEP    = True   # EXP27e sweep, now with the per-K shuffled control
RUN_CLASSMATCH    = True   # EXP27f your class-matched test  <- A vs B is the real read
RUN_POOLED_ORACLE = True   # EXP27g oracle on train+eval     <- comparable to arithmetic EXP9

# CELL D must be re-run: FACT_SPLIT/FACT_MAX changed, so facts.jsonl is rebuilt and the eval
# split (hence the bin) changes. The recon map is re-fit and re-measured in the SAME run, so the
# recon-vs-task comparison stays internally valid even though absolute numbers move.
for _k in ["FACT_SPLIT","FACT_MAX","FACT_MIN_CLASS_COUNT","FACT_RANK","FACT_WD","FACT_LR",
           "FACT_EPOCHS","FACT_MAX_STEPS","FACT_MON_STEPS","FACT_MON_DENSE","FACT_SEEDS"]:
    print(f"  {_k:22s} = {globals()[_k]}")


  FACT_SPLIT             = train
  FACT_MAX               = 40000
  FACT_MIN_CLASS_COUNT   = 2
  FACT_RANK              = 32
  FACT_WD                = 0.01
  FACT_LR                = 0.0001
  FACT_EPOCHS            = 4
  FACT_MAX_STEPS         = 3000
  FACT_MON_STEPS         = 100
  FACT_MON_DENSE         = 300
  FACT_SEEDS             = [0]


In [9]:
# === CELL D: build a LONG-TAIL fact set for EXP27 (run before EXP27) ===
# The builtin capitals list produced an EMPTY bin: the 2B recipient solved 52/52. EXP27 needs
# facts the 9B DONOR gets right and the 2B RECIPIENT gets wrong, which means genuine long tail.
#
# SOURCES
#   "triviaqa"    naturalistic, human-written questions. Best answer to the paper's "our tasks
#                 are synthetic" limitation, and 2B models genuinely struggle on it.
#   "counterfact" ROME's subject-relation-object set. Templated, but very long-tailed.
#   "<path>"      your own JSONL, one {"prompt":..., "answer":...} per line (no build needed).
#
# HONESTY NOTE: I could not verify the HF config string or the CounterFact URL from this machine.
# Each loader PRINTS what it is attempting and raises loudly on failure rather than silently
# yielding a small set. If an identifier is wrong the traceback names it and you can fix it --
# check the dataset card on huggingface.co. Nothing below is silently best-effort.

FACT_BUILD = globals().get("FACT_BUILD", "triviaqa")   # "triviaqa" | "counterfact"
# TriviaQA rc.nocontext: validation is only ~17.9k records, so FACT_MAX above that SILENTLY
# caps. The train split has ~138k. We do our own dedup + train/eval split below, and the
# models never see labels from either, so "train" is just a larger fact pool -- not leakage.
FACT_SPLIT = globals().get("FACT_SPLIT", "train")      # "train" (~138k) | "validation" (~17.9k)
FACT_MAX   = globals().get("FACT_MAX", 40000)          # candidates to write; want >= ~5000
FACT_OUT   = globals().get("FACT_OUT", "facts.jsonl")
FACT_MAX_ANS_WORDS = globals().get("FACT_MAX_ANS_WORDS", 3)   # keep answers short

import json as _json, re as _re, os as _os

# exemplars are generic and deliberately NOT drawn from any of the sources below
_QA_FEWSHOT = ("Question: In which country is the city of Osaka?\nAnswer: Japan\n\n"
               "Question: Who wrote the play Hamlet?\nAnswer: Shakespeare\n\n"
               "Question: What is the chemical symbol for gold?\nAnswer: Au\n\n")

# CounterFact prompts are cloze stems ("The capital of X is"), so they get cloze exemplars
_CLOZE_FEWSHOT = ("Osaka is located in the country of Japan.\n"
                  "The chemical symbol for gold is Au.\n"
                  "Hamlet was written by Shakespeare.\n")

def _ok_answer(a):
    a = (a or "").strip()
    return bool(a) and len(a.split()) <= FACT_MAX_ANS_WORDS and a.isprintable()

rows = []

if FACT_BUILD == "triviaqa":
    print(f"CELL D: attempting  load_dataset('trivia_qa', 'rc.nocontext', split={FACT_SPLIT!r})")
    from datasets import load_dataset
    ds = load_dataset("trivia_qa", "rc.nocontext", split=FACT_SPLIT)
    print(f"  loaded {len(ds)} records | fields: {list(ds.features)[:8]}")
    for r in ds:
        q = (r.get("question") or "").strip()
        ans = r.get("answer") or {}
        a = (ans.get("value") or "").strip() if isinstance(ans, dict) else str(ans).strip()
        if not q or not _ok_answer(a): continue
        rows.append({"prompt": _QA_FEWSHOT + f"Question: {q}\nAnswer:", "answer": a})
        if len(rows) >= FACT_MAX: break

elif FACT_BUILD == "counterfact":
    URL = globals().get("COUNTERFACT_URL",
                        "https://rome.baulab.info/data/dsets/counterfact.json")
    print(f"CELL D: attempting download of CounterFact from {URL}")
    import urllib.request
    _local = "counterfact.json"
    if not _os.path.exists(_local):
        urllib.request.urlretrieve(URL, _local)
    recs = _json.load(open(_local))
    print(f"  loaded {len(recs)} records | first-record keys: {list(recs[0])}")
    for r in recs:
        rw = r.get("requested_rewrite") or {}
        tmpl, subj = rw.get("prompt"), rw.get("subject")
        tgt = (rw.get("target_true") or {}).get("str")
        if not (tmpl and subj and _ok_answer(tgt)): continue
        stem = tmpl.replace("{}", subj).strip()
        rows.append({"prompt": _CLOZE_FEWSHOT + stem, "answer": tgt.strip()})
        if len(rows) >= FACT_MAX: break

else:
    raise ValueError(f"FACT_BUILD must be 'triviaqa' or 'counterfact' "
                     f"(got {FACT_BUILD!r}); to use your own file set FACT_SOURCE directly")

if len(rows) < 500:
    raise RuntimeError(f"only {len(rows)} usable rows from {FACT_BUILD}. That will not give a "
                       f"usable bin. Check the loader above, or raise FACT_MAX / "
                       f"FACT_MAX_ANS_WORDS.")

# de-duplicate on prompt so the train/eval split cannot leak the same fact both ways
_seen, uniq = set(), []
for r in rows:
    if r["prompt"] in _seen: continue
    _seen.add(r["prompt"]); uniq.append(r)

with open(FACT_OUT, "w") as fh:
    for r in uniq:
        fh.write(_json.dumps(r) + "\n")

_alen = sum(len(r["answer"].split()) for r in uniq) / len(uniq)
print(f"\n  wrote {len(uniq)} unique facts -> {FACT_OUT}  (mean answer length {_alen:.2f} words)")
print("  sample:")
for r in uniq[:3]:
    print(f"    prompt tail ...{r['prompt'][-70:]!r}  -> {r['answer']!r}")
print(f"\n  NEXT: set  FACT_SOURCE = {FACT_OUT!r}  and run EXP27.")
print("  EXP27 prints donor/recipient accuracy and the bin size BEFORE it trains anything,")
print("  so if this source is still too easy you will see it in under a minute.")
FACT_SOURCE = FACT_OUT


CELL D: attempting  load_dataset('trivia_qa', 'rc.nocontext', split='train')


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

rc.nocontext/train-00000-of-00001.parque(…):   0%|          | 0.00/55.4M [00:00<?, ?B/s]

rc.nocontext/validation-00000-of-00001.p(…):   0%|          | 0.00/7.34M [00:00<?, ?B/s]

rc.nocontext/test-00000-of-00001.parquet:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/138384 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/17944 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17210 [00:00<?, ? examples/s]

  loaded 138384 records | fields: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer']

  wrote 40000 unique facts -> facts.jsonl  (mean answer length 1.53 words)
  sample:
    prompt tail ...'ican-born Sinclair won the Nobel Prize for Literature in 1930?\nAnswer:'  -> 'Sinclair Lewis'
    prompt tail ...'swer: Au\n\nQuestion: Where in England was Dame Judi Dench born?\nAnswer:'  -> 'York'
    prompt tail ...'e did Billboard magazine first publish and American hit chart?\nAnswer:'  -> '30s'

  NEXT: set  FACT_SOURCE = 'facts.jsonl'  and run EXP27.
  EXP27 prints donor/recipient accuracy and the bin size BEFORE it trains anything,
  so if this source is still too easy you will see it in under a minute.


In [10]:
# === CELL E27b: EXP27 (FIXED) — NATURALISTIC FACTUAL RECALL ===
#
# WHAT WAS WRONG IN THE FIRST RUN, and what changed:
#
# (1) native_full = 0.146 on a bin where native_first is 0 BY CONSTRUCTION. Impossible if the
#     two metrics agreed, so they did not. The bin was built on EXACT FIRST TOKEN ID, but
#     full-answer was scored on the DECODED STRING. A model can reach the same string through a
#     different tokenization (' Paris' as one token vs ' Par' + 'is'), so items failed the token
#     check, entered the bin, and then passed the string check. Arithmetic never shows this
#     because digits tokenize canonically.
#     FIX: the bin is now defined on the DECODED FULL ANSWER for both models -- the same rule the
#     paper's cross-family arms already use. native_full is now 0 by construction; native_first
#     is nonzero and reported, exactly as in the cross-family setting.
#
# (2) `startswith` matching was far too loose: gold "Au" matched "Australia", gold "US" matched
#     "used to be a colony". TriviaQA is full of 2-4 character answers, so this inflated EVERY
#     *_full number in every condition.
#     FIX: strict match -- normalized equality, or gold followed by a word boundary.
#
# (3) recon (0.251) BEAT task (0.146), inverted from every same-family result. Most likely the
#     task map is data-starved: arithmetic has 9 answer classes over 3000 items, TriviaQA has
#     thousands over 4800. The cell now PRINTS the distinct-answer-token count, prints per-epoch
#     loss, and periodically prints held-out conferral so underfitting is visible while it runs.
#
# REUSES from the setup cells above: states_and_top, left_pad, fit_ridge, patch_vec_batch,
# _graft, wilson_bools, across_seed_ci, fmt, L2_SINGLE, L9_SINGLE, ARITH_BATCH, DEVICE, RESULTS,
# tokenizer, model_2b, model_9b.

RUN_FACTS       = globals().get("RUN_FACTS", True)
FACT_SOURCE     = globals().get("FACT_SOURCE", "facts.jsonl")
FACT_TRAIN_FRAC = globals().get("FACT_TRAIN_FRAC", 0.6)
FACT_SEEDS      = globals().get("FACT_SEEDS", [0, 1, 2, 3, 4])
FACT_EPOCHS     = globals().get("FACT_EPOCHS", 4)      # held-out conferral peaked at epoch 2
#                 more epochs OVERFIT: train CE -> 0.15 while held-out fell. Do not raise this
#                 to fix underperformance; fix examples-per-class in CELL D instead.
# The v3 fix. The previous run ended with best_step == 0 on all five seeds: full-rank Adam at
# lr=1e-3 drove TRAIN conferral 0.33 -> 0.99 inside one epoch while held-out sat at 0.13-0.19,
# so FACT_KEEP_BEST restored the recon init and every "task" number in the JSON was the recon
# map. Three changes, all aimed at "can it learn a GENERAL read-out rather than memorize":
#   FACT_RANK  low-rank residual on the recon init: W_eff = FWr + A@B, B initialized to ZERO so
#              step 0 is still exactly the recon map. Rank 32 cannot store thousands of
#              fact-specific directions, so the only way to lower CE is a general correction.
#              Set FACT_RANK = 0 to recover the old full-rank behaviour for an A/B.
#   FACT_LR    1e-3 -> 1e-4. Train conferral hit 0.94 by step 400; that is far too fast.
#   FACT_WD    weight decay on A,B only. Decaying them toward 0 IS anchoring to the recon map,
#              which is the right prior here. Never applied to the bias.
FACT_RANK       = globals().get("FACT_RANK", 32)       # 0 = full-rank (old behaviour)
FACT_WD         = globals().get("FACT_WD", 1e-2)       # weight decay on the low-rank residual
FACT_LR         = globals().get("FACT_LR", 1e-4)
# Train-only class filter. Singleton answer classes cannot teach a generalizable read-out --
# one example is pure memorization fuel. This drops them from the CE loop ONLY. The ridge/recon
# map is still fit on ALL of FT_TRAIN (line ~146) and FT_EVAL/the bin are untouched, so the
# recon baseline stays comparable to the previous run. Set 1 to disable.
FACT_MIN_CLASS_COUNT = globals().get("FACT_MIN_CLASS_COUNT", 2)
FACT_EVAL_EVERY = globals().get("FACT_EVAL_EVERY", 2)  # print held-out conferral every N epochs
FACT_MON_N      = globals().get("FACT_MON_N", 160)     # items used for that quick monitor
# The task map is warm-started AT the recon map (W=FWr, b=Fmu2), so it begins at recon's score
# and can only be judged by whether training moves it UP. Epoch-level monitoring was far too
# coarse: one epoch here is 2250 Adam steps (the arithmetic map takes 1128 steps IN TOTAL), so
# the first measurement already came long after any optimum. Monitor by STEP instead.
FACT_MON_STEPS  = globals().get("FACT_MON_STEPS", 100)   # held-out check every N optimizer steps
FACT_MON_DENSE  = globals().get("FACT_MON_DENSE", 300)   # ...every 25 steps for the first N
FACT_MAX_STEPS  = globals().get("FACT_MAX_STEPS", 3000)  # hard cap on total steps per seed
FACT_KEEP_BEST  = globals().get("FACT_KEEP_BEST", True)  # keep the best-held-out checkpoint
MAX_NEW_FACT    = globals().get("MAX_NEW_FACT", 12)
FACT_MAX_EVAL   = globals().get("FACT_MAX_EVAL", 3000) # cap: bin build needs generation x2 models

# Guard: every knob above is a globals().get() default, so a value left in the kernel by an
# earlier run silently wins. That has already happened three times in this experiment. The RUN
# CONFIG cell hard-assigns them and sets this sentinel; without it, stop rather than burn hours
# on stale settings. Then echo what is ACTUALLY in effect -- one readout, no inference.
assert globals().get("RUN_CONFIG_APPLIED") == "v3", (
    "RUN CONFIG cell was not run (it sits just above CELL D). Run it first: without it this cell "
    "would reuse FACT_MAX/FACT_LR/FACT_MAX_STEPS/FACT_SEEDS left over from your previous run.")
print("EXP27b running with:  seeds =", FACT_SEEDS, " rank =", FACT_RANK, " lr =", FACT_LR,
      " wd =", FACT_WD, " epochs =", FACT_EPOCHS, " max_steps =", FACT_MAX_STEPS,
      " min_class_count =", FACT_MIN_CLASS_COUNT)

FACT_OK = False
if RUN_FACTS:
    import json as _json

    def _norm(t):
        return "".join(c for c in t.lower() if c.isascii() and (c.isalnum() or c == " ")).strip()

    def _match(gen_text, gold):
        """Strict: normalized equality, or gold followed by a word boundary.
        Rejects the old failure mode where gold 'Au' matched 'Australia'."""
        g, p = _norm(gold), _norm(gen_text.split("\n")[0])
        return bool(g) and (p == g or p.startswith(g + " "))

    def _first_answer_token(tok, prompt, answer):
        p = tok(prompt).input_ids
        f = tok(prompt + " " + answer).input_ids
        if f[:len(p)] != p or len(f) <= len(p):
            return None, None
        j = len(p)
        while j < len(f) and tok.decode([f[j]]).strip() == "":
            j += 1
        return (torch.tensor(f[:j]), f[j]) if j < len(f) else (None, None)

    _raw = []
    with open(FACT_SOURCE) as fh:
        for line in fh:
            line = line.strip()
            if line: _raw.append(_json.loads(line))
    FACTS = []
    for r in _raw:
        ids, tid = _first_answer_token(tokenizer, r["prompt"], r["answer"])
        if tid is not None:
            FACTS.append(dict(prompt=r["prompt"], answer=r["answer"], ids=ids, tok=tid))
    print(f"E27b: {len(_raw)} candidates -> {len(FACTS)} encoded")

    _fi = list(range(len(FACTS))); random.Random(0).shuffle(_fi)
    _cut = max(1, int(len(_fi) * FACT_TRAIN_FRAC))
    FT_TRAIN = [FACTS[i] for i in _fi[:_cut]]
    FT_EVAL  = [FACTS[i] for i in _fi[_cut:]][:FACT_MAX_EVAL]
    _ntok = len({p["tok"] for p in FT_TRAIN})
    print(f"      train={len(FT_TRAIN)} eval={len(FT_EVAL)}")
    print(f"      DISTINCT ANSWER TOKENS IN TRAIN = {_ntok}  "
          f"({len(FT_TRAIN)/max(1,_ntok):.1f} examples per class)")
    if _ntok > len(FT_TRAIN) / 20:
        print("      >>> WARNING: very few examples per answer class. A cross-entropy task map")
        print("      >>> may be data-starved here; watch the per-epoch loss and monitor below.")
    FACT_OK = len(FT_EVAL) >= 50

if FACT_OK:
    @torch.inference_mode()
    def _generate(model, items, vec_fn=None, layer=None):
        """Free-generate for `items`. If vec_fn is given, graft vec_fn(batch_slice) first."""
        outs, h = [], None
        if vec_fn is not None:
            h = model.model.layers[layer].register_forward_hook(patch_vec_batch)
        try:
            for i in range(0, len(items), ARITH_BATCH):
                chunk = items[i:i + ARITH_BATCH]
                ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
                if vec_fn is not None:
                    v = vec_fn(i, len(chunk))
                    assert v.shape[0] == len(chunk), f"graft {v.shape[0]} != {len(chunk)}"
                    _graft["vec"] = v
                gen = model.generate(ids.to(DEVICE), attention_mask=m.to(DEVICE),
                                     max_new_tokens=MAX_NEW_FACT, do_sample=False,
                                     pad_token_id=tokenizer.eos_token_id)
                outs += tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
        finally:
            if h is not None: h.remove()
            _graft["vec"] = None
        return outs

    # ---- BIN: decoded full answer, both models. native_full is now 0 by construction. ----
    print("      building bin by GENERATION (both models) — this is the slow part ...")
    _d_txt = _generate(model_9b, FT_EVAL)
    _r_txt = _generate(model_2b, FT_EVAL)
    F_donor_ok = [_match(_d_txt[i], FT_EVAL[i]["answer"]) for i in range(len(FT_EVAL))]
    F_recip_ok = [_match(_r_txt[i], FT_EVAL[i]["answer"]) for i in range(len(FT_EVAL))]
    F_UNSOLV = [i for i in range(len(FT_EVAL)) if F_donor_ok[i] and not F_recip_ok[i]]
    F_SOLV   = [i for i in range(len(FT_EVAL)) if F_donor_ok[i] and F_recip_ok[i]]
    print(f"      donor {sum(F_donor_ok)}/{len(FT_EVAL)} | recipient {sum(F_recip_ok)}/{len(FT_EVAL)}")
    print(f"      >>> UNSOLVABLE BIN n = {len(F_UNSOLV)}")

    # native FIRST-token on this bin is NOT zero (bin is full-answer defined) — report it
    _, F2_top = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in FT_EVAL])
    F_nat_first = [F2_top[j] == FT_EVAL[j]["tok"] for j in F_UNSOLV]
    F_nat_full  = [_match(_r_txt[j], FT_EVAL[j]["answer"]) for j in F_UNSOLV]   # 0 by construction
    print(f"      native on bin: first={sum(F_nat_first)/max(1,len(F_UNSOLV)):.3f} "
          f"full={sum(F_nat_full)/max(1,len(F_UNSOLV)):.3f} (full must be 0.000)")
    FACT_OK = len(F_UNSOLV) >= 50
    if not FACT_OK: print("EXP27b STOPPED: bin < 50.")

if FACT_OK:
    FX9t, _ = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in FT_TRAIN]); FX9t = FX9t[L9_SINGLE]
    FX2t, _ = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in FT_TRAIN]); FX2t = FX2t[L2_SINGLE]
    FX9e, _ = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in FT_EVAL]);  FX9e = FX9e[L9_SINGLE]
    FX2e, _ = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in FT_EVAL]);  FX2e = FX2e[L2_SINGLE]
    Fmu9, Fmu2, FWr = fit_ridge(FX9t, FX2t)
    Fmu9d, Fmu2d, FWr_d = Fmu9.to(DEVICE), Fmu2.to(DEVICE), FWr.to(DEVICE)

    @torch.inference_mode()
    def F_first(vec_fn, idxs):
        h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
        out = []
        try:
            for i in range(0, len(idxs), ARITH_BATCH):
                sub = idxs[i:i + ARITH_BATCH]
                v = vec_fn(sub); assert v.shape[0] == len(sub)
                _graft["vec"] = v
                ids, m = left_pad([FT_EVAL[j]["ids"] for j in sub], tokenizer.pad_token_id)
                top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
                out += [top[k] == FT_EVAL[sub[k]]["tok"] for k in range(len(sub))]
        finally:
            h.remove(); _graft["vec"] = None
        return out

    def F_full(vec_fn, idxs):
        items = [FT_EVAL[j] for j in idxs]
        def _vf(i, n): return vec_fn(idxs[i:i + n])
        txt = _generate(model_2b, items, vec_fn=_vf, layer=L2_SINGLE)
        return [_match(txt[k], items[k]["answer"]) for k in range(len(items))]

    MON = F_UNSOLV[:FACT_MON_N]

    # ---- task-map training subset (CE loop only; ridge and bin untouched) ----
    from collections import Counter as _Counter
    _cc = _Counter(p["tok"] for p in FT_TRAIN)
    FT_TASK_IDX = [i for i, p in enumerate(FT_TRAIN) if _cc[p["tok"]] >= FACT_MIN_CLASS_COUNT]
    _ncls_task = len({FT_TRAIN[i]["tok"] for i in FT_TASK_IDX})
    _epc_task = len(FT_TASK_IDX) / max(1, _ncls_task)
    print(f"      CE training subset: {len(FT_TASK_IDX)}/{len(FT_TRAIN)} items over "
          f"{_ncls_task} classes ({_epc_task:.1f} per class), min class count "
          f"= {FACT_MIN_CLASS_COUNT}  [was {len(FT_TRAIN)/max(1,_ntok):.1f} per class]")
    print(f"      ridge/recon map still fit on ALL {len(FT_TRAIN)} items; bin unchanged")
    assert len(FT_TASK_IDX) >= 200, (
        f"only {len(FT_TASK_IDX)} items survive FACT_MIN_CLASS_COUNT="
        f"{FACT_MIN_CLASS_COUNT}; lower it or raise FACT_MAX in CELL D")

    # Memorization probe: the same measurement on TRAINING items the map has already seen.
    # If train conferral climbs while held-out falls, the map is memorizing fact-specific
    # directions rather than learning a general read-out. That is the decisive test for
    # "what is it converging to".
    _TRMON = FT_TASK_IDX[:FACT_MON_N]   # probe items the CE loop actually trains on
    @torch.inference_mode()
    def F_first_train(W_, b_):
        h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
        out = []
        try:
            for i in range(0, len(_TRMON), ARITH_BATCH):
                sub = _TRMON[i:i + ARITH_BATCH]
                _graft["vec"] = (FX9t[sub].to(DEVICE) - Fmu9d) @ W_ + b_
                ids, m = left_pad([FT_TRAIN[j]["ids"] for j in sub], tokenizer.pad_token_id)
                top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
                out += [top[k] == FT_TRAIN[sub[k]]["tok"] for k in range(len(sub))]
        finally:
            h.remove(); _graft["vec"] = None
        return sum(out) / max(1, len(out))
    F_task_maps  = []
    F_best_step  = []          # per seed; 0 means training never beat the recon init
    _d9, _d2 = FWr.shape
    for sd in FACT_SEEDS:
        torch.manual_seed(sd)
        b = Fmu2.clone().to(DEVICE).float().requires_grad_(True)
        if FACT_RANK and FACT_RANK > 0:
            # LoRA-style init: A ~ small random, B = 0  =>  W_eff == FWr exactly at step 0, so the
            # first monitor row is still the recon map and "did training help" stays readable.
            A = (torch.randn(_d9, FACT_RANK, device=DEVICE) / math.sqrt(_d9)).requires_grad_(True)
            B = torch.zeros(FACT_RANK, _d2, device=DEVICE).requires_grad_(True)
            groups = [{"params": [A, B], "weight_decay": FACT_WD},
                      {"params": [b],    "weight_decay": 0.0}]
            def _W_eff(): return FWr_d + A @ B
            def _vec(x):
                # never materialize the 3584x2304 product per step: [16,d9]->[16,r]->[16,d2]
                xc = x - Fmu9d
                return xc @ FWr_d + (xc @ A) @ B + b
        else:
            Wf = FWr.clone().to(DEVICE).requires_grad_(True)
            groups = [{"params": [Wf, b], "weight_decay": 0.0}]
            def _W_eff(): return Wf
            def _vec(x): return (x - Fmu9d) @ Wf + b
        opt = torch.optim.Adam(groups, lr=FACT_LR)
        model_2b.requires_grad_(False)
        h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
        try:
            idx = list(FT_TASK_IDX)
            gstep, stop = 0, False
            best_W, best_b, best_step = _W_eff().detach().clone(), b.detach().clone(), 0
            # step 0 = the recon map itself, so the very first row is the baseline to beat
            _m0 = F_first(lambda sub: (FX9e[sub].to(DEVICE) - Fmu9d) @ best_W + best_b, MON)
            best_acc = sum(_m0) / len(_m0)
            print(f"      seed {sd} step     0 (= recon init)  TRAIN = {F_first_train(best_W, best_b):.3f}"
                  f"   held-out = {best_acc:.3f}   |dW|/|W| = 0.000", flush=True)
            for ep in range(FACT_EPOCHS):
                if stop: break
                random.Random(100 * sd + ep).shuffle(idx)
                for st in range(0, len(idx), ARITH_BATCH):
                    sub = idx[st:st + ARITH_BATCH]
                    _graft["vec"] = _vec(FX9t[sub].to(DEVICE))
                    ids, m = left_pad([FT_TRAIN[k]["ids"] for k in sub], tokenizer.pad_token_id)
                    logits = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                    tgt = torch.tensor([FT_TRAIN[k]["tok"] for k in sub], device=DEVICE)
                    loss = F.cross_entropy(logits, tgt)
                    opt.zero_grad(); loss.backward(); opt.step()
                    gstep += 1
                    # dense early monitoring: the previous run's first measurement landed at step
                    # 100, by which point TRAIN conferral was already 0.39 and climbing fast.
                    if (gstep % FACT_MON_STEPS == 0) or (gstep <= FACT_MON_DENSE and gstep % 25 == 0):
                        _Wd, _bd = _W_eff().detach().clone(), b.detach().clone()
                        _mon = F_first(lambda sub: (FX9e[sub].to(DEVICE) - Fmu9d) @ _Wd + _bd, MON)
                        _acc = sum(_mon) / len(_mon)
                        flag = ""
                        if _acc > best_acc:
                            best_acc, best_W, best_b, best_step = _acc, _Wd, _bd, gstep
                            flag = "  <- best"
                        _tr = F_first_train(_Wd, _bd)
                        _dw = float((_Wd - FWr_d).norm() / FWr_d.norm())
                        print(f"      seed {sd} step {gstep:5d} (ep {ep})  CE = {float(loss.item()):.4f}"
                              f"   TRAIN = {_tr:.3f}   held-out = {_acc:.3f}"
                              f"   |dW|/|W| = {_dw:.3f}{flag}", flush=True)
                    if gstep >= FACT_MAX_STEPS:
                        stop = True; break
            print(f"      seed {sd} BEST held-out = {best_acc:.3f} at step {best_step}"
                  f"{'  (never beat the recon init -> this seed IS the recon map)' if best_step == 0 else ''}",
                  flush=True)
            if FACT_KEEP_BEST:
                W_out, b_out = best_W, best_b
            else:
                W_out, b_out = _W_eff().detach().clone(), b.detach().clone()
        finally:
            h.remove(); _graft["vec"] = None
            model_2b.requires_grad_(True)
        F_best_step.append(best_step)
        F_task_maps.append((W_out.detach().clone(), b_out.detach().clone()))

    # Guard against the previous run's reporting hazard: when best_step == 0 the kept checkpoint
    # is literally the recon init, so every "task_*" number below is the recon map wearing a task
    # label (that is why the last run printed five identical per-seed values and an across-seed
    # interval of [0.217, 0.217]). Make it impossible to read past.
    F_TASK_IS_RECON = [bs == 0 for bs in F_best_step]
    if all(F_TASK_IS_RECON):
        print("\n      " + "!" * 68)
        print("      !! NO SEED BEAT THE RECON INIT. Every task_* number below is the recon")
        print("      !! map, not a trained task map. Do not report them as a task map.")
        print("      " + "!" * 68 + "\n", flush=True)

    F_recon = lambda sub: (FX9e[sub].to(DEVICE) - Fmu9d) @ FWr_d + Fmu2d
    def F_task(i):
        W, b = F_task_maps[i]
        return lambda sub: (FX9e[sub].to(DEVICE) - Fmu9d) @ W + b
    F_self  = lambda sub: FX2e[sub].to(DEVICE).float()
    def _fpartner(idxs, seed):
        rng = random.Random(seed); pool = list(idxs); perm = pool[:]
        for _ in range(200):
            rng.shuffle(perm)
            if all(FT_EVAL[perm[k]]["tok"] != FT_EVAL[pool[k]]["tok"] for k in range(len(pool))):
                return dict(zip(pool, perm)), 0
        return dict(zip(pool, perm)), sum(
            1 for k in range(len(pool)) if FT_EVAL[perm[k]]["tok"] == FT_EVAL[pool[k]]["tok"])
    F_PART, _fcol = _fpartner(F_UNSOLV, 7)
    def F_shuf(sub):
        W0, b0 = F_task_maps[0]
        return (FX9e[[F_PART[j] for j in sub]].to(DEVICE) - Fmu9d) @ W0 + b0

    # RECON's own shuffled control. Without this we cannot distinguish two explanations for
    # recon lifting first-token 0.061 -> 0.289:
    #   (a) it transfers this problem's donor content  -> shuffled recon should COLLAPSE
    #   (b) writing any well-formed, on-manifold vector papers over the recipient's errors
    #       -> shuffled recon should lift it just as much
    # (b) is the "it is only fixing minor glitches" hypothesis. This is the test that settles it.
    def F_recon_shuf(sub):
        return (FX9e[[F_PART[j] for j in sub]].to(DEVICE) - Fmu9d) @ FWr_d + Fmu2d

    # Exact two-sided McNemar on discordant pairs. recon vs task are scored on the SAME 570
    # items, so the marginal Wilson intervals overlapping is not the right test -- the paired
    # one is. Fraction keeps the tail exact (plain floats overflow past n ~ 1030).
    from fractions import Fraction as _Fr
    def _mcnemar(a_ok, b_ok):
        """a_ok/b_ok: aligned bool lists. Returns (a_only, b_only, exact two-sided p)."""
        bb = sum(1 for x, y in zip(a_ok, b_ok) if x and not y)
        cc = sum(1 for x, y in zip(a_ok, b_ok) if y and not x)
        n = bb + cc
        if n == 0: return bb, cc, 1.0
        k = min(bb, cc)
        tail = sum(_Fr(math.comb(n, i)) for i in range(k + 1)) / _Fr(2) ** n
        return bb, cc, min(1.0, float(2 * tail))

    # Evaluate every condition ONCE, keep the per-item bools, build the summary from them.
    # (The previous version scored task(0) twice -- once in _seed_first, once for the JSON.)
    _B_self   = F_first(F_self,       F_UNSOLV)
    _B_shuf   = F_first(F_shuf,       F_UNSOLV)
    _B_recon  = F_first(F_recon,      F_UNSOLV)
    _B_rshuf  = F_first(F_recon_shuf, F_UNSOLV)
    _B_task   = [F_first(F_task(i),   F_UNSOLV) for i in range(len(F_task_maps))]
    _seed_first = [sum(bl) / len(F_UNSOLV) for bl in _B_task]
    _B_recon_f = F_full(F_recon,    F_UNSOLV)
    _B_task_f  = F_full(F_task(0),  F_UNSOLV)

    # Selection-disjoint slice: MON = F_UNSOLV[:FACT_MON_N] was used to pick the checkpoint, and
    # it is a SUBSET of the reported bin, so task_first is selected partly on its own test set.
    # These items were never seen by the selection, so recon-vs-task is like-for-like on them.
    _DISJ = list(range(FACT_MON_N, len(F_UNSOLV)))
    _b_r, _b_t, _p_first = _mcnemar(_B_recon, _B_task[0])
    _f_r, _f_t, _p_full  = _mcnemar(_B_recon_f, _B_task_f)

    RESULTS["factual_recall_v2"] = {
        "_what": ("naturalistic factual recall. BIN IS DEFINED ON THE DECODED FULL ANSWER for "
                  "both models (same rule as the paper's cross-family arms), so native full-answer "
                  "is 0 by construction and native FIRST-token is nonzero."),
        "_headline": ("first-token favours the TASK map and full-answer favours the RECON map, by "
                      "~10 items out of the bin either way. Read full-answer as primary: the task "
                      "map is trained with CE on the gold first-token id, so first-token is its "
                      "own training objective, and checkpoint selection used MON, a subset of this "
                      "bin. See *_selection_disjoint and mcnemar_* before claiming either wins."),
        "_fixes": ["bin now full-answer defined (was first-token, which disagreed with the "
                   "full-answer metric via tokenization non-determinism)",
                   "strict answer matching (was startswith: 'Au' matched 'Australia')",
                   "per-epoch loss + held-out conferral monitor",
                   f"v3: low-rank residual on the recon init (rank {FACT_RANK}), lr {FACT_LR}, "
                   f"wd {FACT_WD}, singleton answer classes dropped from the CE loop only",
                   "v4: paired McNemar recon-vs-task, and a selection-disjoint slice",
                   f"epochs {FACT_EPOCHS}"],
        "source": FACT_SOURCE, "n_encoded": len(FACTS),
        "n_train": len(FT_TRAIN), "n_eval": len(FT_EVAL),
        "distinct_answer_tokens_in_train": _ntok,
        "examples_per_answer_class": round(len(FT_TRAIN) / max(1, _ntok), 2),
        "task_map_cfg": {"rank": FACT_RANK, "lr": FACT_LR, "weight_decay": FACT_WD,
                         "min_class_count": FACT_MIN_CLASS_COUNT, "epochs": FACT_EPOCHS,
                         "max_steps": FACT_MAX_STEPS, "keep_best": FACT_KEEP_BEST},
        "ce_train_items": len(FT_TASK_IDX), "ce_train_classes": _ncls_task,
        "ce_examples_per_class": round(_epc_task, 2),
        "donor_full_acc": round(sum(F_donor_ok) / len(FT_EVAL), 4),
        "recipient_full_acc": round(sum(F_recip_ok) / len(FT_EVAL), 4),
        "n_unsolv": len(F_UNSOLV), "n_solv": len(F_SOLV),
        "native_first":    fmt(wilson_bools(F_nat_first)),
        "native_full":     fmt(wilson_bools(F_nat_full)),
        "selfgraft_first": fmt(wilson_bools(_B_self)),
        "_selfgraft_note": ("grafting the recipient's own captured state back in. Should equal "
                            "native; a 1-2 item gap is the fp32 capture -> bf16 graft round-trip, "
                            "not a failure of the control"),
        "shuffle_first":   fmt(wilson_bools(_B_shuf)),
        "_shuffle_note": ("shuffle_first uses F_task_maps[0] and recon_SHUFFLED_first uses the "
                          "ridge map. When task_is_recon_init[0] is false these are two "
                          "INDEPENDENT controls and their agreement is real corroboration; when "
                          "it is true they are the same computation and only one control exists"),
        "recon_first":     fmt(wilson_bools(_B_recon)),
        "recon_SHUFFLED_first": fmt(wilson_bools(_B_rshuf)),
        "_recon_shuffle_note": ("if this is near native the recon lift is problem-specific "
                                "conferral; if it is near recon_first the lift is generic, i.e. "
                                "writing any on-manifold vector repairs the recipient"),
        "task_best_step_per_seed": F_best_step,
        "task_is_recon_init_per_seed": F_TASK_IS_RECON,
        "_task_warning": ("ALL SEEDS ARE THE RECON INIT -- every task_* field below is the recon "
                          "map, not a trained task map; do not report them as one"
                          if all(F_TASK_IS_RECON) else
                          "some seeds improved on the recon init; see task_best_step_per_seed"),
        "task_first_seed0": fmt(wilson_bools(_B_task[0])),
        "task_first_per_seed": [round(v, 4) for v in _seed_first],
        "task_first_acrossseed": fmt(across_seed_ci(_seed_first)) if len(_seed_first) > 1 else "n/a",
        "recon_full":       fmt(wilson_bools(_B_recon_f)),
        "task_full_seed0":  fmt(wilson_bools(_B_task_f)),
        # --- paired tests: the same 570 items, so marginal CIs are the wrong comparison ---
        "mcnemar_first_recon_vs_task": {
            "recon_only": _b_r, "task_only": _b_t, "p_exact_two_sided": round(_p_first, 5),
            "reading": "recon_only = solved by recon and missed by task, and vice versa"},
        "mcnemar_full_recon_vs_task": {
            "recon_only": _f_r, "task_only": _f_t, "p_exact_two_sided": round(_p_full, 5)},
        # --- selection-disjoint: items never used to pick the checkpoint ---
        "n_selection_disjoint": len(_DISJ),
        "recon_first_selection_disjoint": fmt(wilson_bools([_B_recon[j] for j in _DISJ])),
        "task_first_selection_disjoint":  fmt(wilson_bools([_B_task[0][j] for j in _DISJ])),
        "shuffled_pairing": {"n_pairs": len(F_PART), "unavoidable_same_token": _fcol},
    }
    print("EXP27b:", json.dumps(RESULTS["factual_recall_v2"], indent=2))
elif RUN_FACTS:
    print("EXP27b did not run to completion — see the stop message above.")


EXP27b running with:  seeds = [0]  rank = 32  lr = 0.0001  wd = 0.01  epochs = 4  max_steps = 3000  min_class_count = 2
E27b: 40000 candidates -> 40000 encoded
      train=24000 eval=3000
      DISTINCT ANSWER TOKENS IN TRAIN = 7252  (3.3 examples per class)
      >>> WARNING: very few examples per answer class. A cross-entropy task map
      >>> may be data-starved here; watch the per-epoch loss and monitor below.
      building bin by GENERATION (both models) — this is the slow part ...
      donor 1976/3000 | recipient 1505/3000
      >>> UNSOLVABLE BIN n = 570
      native on bin: first=0.065 full=0.000 (full must be 0.000)


[W911 17:24:55.708826592 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1507852288 bytes (free: 548798464, total: 47665709056).


      CE training subset: 20348/24000 items over 3600 classes (5.7 per class), min class count = 2  [was 3.3 per class]
      ridge/recon map still fit on ALL 24000 items; bin unchanged
      seed 0 step     0 (= recon init)  TRAIN = 0.556   held-out = 0.287   |dW|/|W| = 0.000
      seed 0 step    25 (ep 0)  CE = 1.9880   TRAIN = 0.556   held-out = 0.287   |dW|/|W| = 0.004
      seed 0 step    50 (ep 0)  CE = 3.1938   TRAIN = 0.562   held-out = 0.287   |dW|/|W| = 0.006
      seed 0 step    75 (ep 0)  CE = 1.7337   TRAIN = 0.562   held-out = 0.281   |dW|/|W| = 0.008
      seed 0 step   100 (ep 0)  CE = 2.7095   TRAIN = 0.562   held-out = 0.275   |dW|/|W| = 0.011
      seed 0 step   125 (ep 0)  CE = 3.5128   TRAIN = 0.562   held-out = 0.300   |dW|/|W| = 0.014  <- best
      seed 0 step   150 (ep 0)  CE = 2.3591   TRAIN = 0.556   held-out = 0.275   |dW|/|W| = 0.018
      seed 0 step   175 (ep 0)  CE = 3.4155   TRAIN = 0.562   held-out = 0.294   |dW|/|W| = 0.020
      seed 0 step   200 (ep

In [11]:
# EXP27e — class-count sweep
if not globals().get("RUN_CLASSSWEEP", True):
    print("EXP27e — class-count sweep skipped (RUN_CLASSSWEEP = False)")
else:
    # === EXP27e: CLASS-COUNT SWEEP — the controlled test of "too many classes to learn" ===
    # Paste as a new cell after the 5-seed run. Reuses FX9t / FX9e / FWr / Fmu9d / Fmu2 / FT_TRAIN /
    # FT_EVAL / F_UNSOLV / F_first / _mcnemar from the live kernel.
    #
    # THE CONFOUND THIS REMOVES. Restricting to frequent answers changes two things at once: the
    # number of classes AND the number of training items. To isolate class count we fix the item
    # budget to N_FIX -- the largest budget every condition can supply -- and subsample each
    # condition down to it. So K=50 and K=3200 see the SAME number of examples; only the number of
    # distinct answers among them differs. examples-per-class is printed so the intended gradient
    # is visible.
    #
    # WHY RECON IS THE CONTROL, PER CONDITION. Each K has its own eval subset (bin items whose gold
    # answer is in the top-K classes), so raw conferral is NOT comparable across K -- the subsets
    # differ in size and difficulty. The recon map is scored on the SAME items, so the reported
    # delta (task - recon) is comparable across conditions. That delta is the whole experiment.
    #
    # THE PREDICTION UNDER YOUR THEORY: delta > 0 at small K, falling and going negative as K grows.
    # A flat delta near zero says class count is not the binding constraint.
    for _n in ["FX9t","FX9e","FWr","FWr_d","Fmu9d","Fmu2","Fmu2d","FT_TRAIN","FT_EVAL",
               "F_UNSOLV","F_first","_mcnemar","FACT_RANK","FACT_LR","FACT_WD"]:
        assert _n in globals(), f"{_n} missing -- run the EXP27b cell (v4) first, same kernel"

    from collections import Counter as _C
    K_SWEEP    = globals().get("K_SWEEP", [50, 200, 800, 3200])
    K_EPOCHS   = globals().get("K_EPOCHS", 8)
    K_MIN_EVAL = globals().get("K_MIN_EVAL", 40)     # skip a K whose bin subset is too small to read

    _tr_cnt  = _C(p["tok"] for p in FT_TRAIN)
    _ranked  = [t for t, _ in _tr_cnt.most_common()]
    _bin_cnt = _C(FT_EVAL[j]["tok"] for j in F_UNSOLV)
    print(f"EXP27e  train: {len(FT_TRAIN)} items / {len(_tr_cnt)} classes")
    print(f"        bin:   {len(F_UNSOLV)} items / {len(_bin_cnt)} distinct answers "
          f"(top answer covers {_bin_cnt.most_common(1)[0][1]}/{len(F_UNSOLV)})")

    # item budget every condition can meet
    _pools = {}
    for K in K_SWEEP:
        keep = set(_ranked[:K])
        _pools[K] = [i for i, p in enumerate(FT_TRAIN) if p["tok"] in keep]
    N_FIX = min(len(v) for v in _pools.values())
    print(f"        fixed item budget N_FIX = {N_FIX} "
          f"(smallest pool across K={K_SWEEP}); every condition subsampled to this\n")

    def _train(idx_list, seed):
        torch.manual_seed(seed)
        d9, d2 = FWr.shape
        b = Fmu2.clone().to(DEVICE).float().requires_grad_(True)
        A = (torch.randn(d9, FACT_RANK, device=DEVICE) / math.sqrt(d9)).requires_grad_(True)
        B = torch.zeros(FACT_RANK, d2, device=DEVICE).requires_grad_(True)
        opt = torch.optim.Adam([{"params": [A, B], "weight_decay": FACT_WD},
                                {"params": [b], "weight_decay": 0.0}], lr=FACT_LR)
        model_2b.requires_grad_(False)
        h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
        try:
            idx, ce = list(idx_list), float("nan")
            for ep in range(K_EPOCHS):
                random.Random(1000 * seed + ep).shuffle(idx)
                tot = n = 0
                for s in range(0, len(idx), ARITH_BATCH):
                    sub = idx[s:s + ARITH_BATCH]
                    xc = FX9t[sub].to(DEVICE) - Fmu9d
                    _graft["vec"] = xc @ FWr_d + (xc @ A) @ B + b
                    ids, m = left_pad([FT_TRAIN[k]["ids"] for k in sub], tokenizer.pad_token_id)
                    lg = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                    tgt = torch.tensor([FT_TRAIN[k]["tok"] for k in sub], device=DEVICE)
                    loss = F.cross_entropy(lg, tgt)
                    opt.zero_grad(); loss.backward(); opt.step()
                    tot += float(loss.item()); n += 1
                ce = tot / max(1, n)
        finally:
            h.remove(); _graft["vec"] = None; model_2b.requires_grad_(True)
        return (FWr_d + A @ B).detach().clone(), b.detach().clone(), ce

    SWEEP = []
    print(f"{'K':>6}{'train n':>9}{'ex/class':>10}{'eval n':>8}{'recon':>8}{'task':>8}"
          f"{'delta':>8}{'shuf':>8}{'p':>9}{'final CE':>10}")
    for K in K_SWEEP:
        keep = set(_ranked[:K])
        ev = [j for j in F_UNSOLV if FT_EVAL[j]["tok"] in keep]
        if len(ev) < K_MIN_EVAL:
            print(f"{K:>6}{'-':>9}{'-':>10}{len(ev):>8}   SKIPPED: bin subset < {K_MIN_EVAL}")
            continue
        pool = list(_pools[K]); random.Random(7).shuffle(pool); pool = pool[:N_FIX]
        ncls = len({FT_TRAIN[i]["tok"] for i in pool})
        W, b, ce = _train(pool, 0)
        t_ok = F_first(lambda sub: (FX9e[sub].to(DEVICE) - Fmu9d) @ W + b, ev)
        r_ok = F_first(lambda sub: (FX9e[sub].to(DEVICE) - Fmu9d) @ FWr_d + Fmu2d, ev)
        t, r = sum(t_ok) / len(ev), sum(r_ok) / len(ev)
        _rr, _tt, p = _mcnemar(r_ok, t_ok)
        # SHUFFLED CONTROL, per K. At small K the map trains on few output tokens and is scored
        # only on items answered by those same tokens, so it gains a restricted-output prior the
        # recon map never gets. Feeding it a MISMATCHED donor state removes the content but keeps
        # the prior: near 0.004 (the main shuffle) means the gain is real content; anything much
        # higher is prior, and the delta above is not a clean read of class count.
        _rg = random.Random(31); _pp = list(ev)
        for _ in range(200):
            _rg.shuffle(_pp)
            if all(FT_EVAL[_pp[k]]["tok"] != FT_EVAL[ev[k]]["tok"] for k in range(len(ev))): break
        _PMAP = dict(zip(ev, _pp))
        s_ok = F_first(lambda sub: (FX9e[[_PMAP[j] for j in sub]].to(DEVICE) - Fmu9d) @ W + b, ev)
        sh = sum(s_ok) / len(ev)
        SWEEP.append({"K": K, "train_items": len(pool), "train_classes": ncls,
                      "examples_per_class": round(len(pool) / max(1, ncls), 2),
                      "eval_n": len(ev), "recon_first": round(r, 4), "task_first": round(t, 4),
                      "delta_task_minus_recon": round(t - r, 4),
                      "task_SHUFFLED_first": round(sh, 4),
                      "_shuffle_read": ("near the main shuffle (0.004) => the delta is real "
                                        "content; much higher => restricted-output prior"),
                      "mcnemar": {"recon_only": _rr, "task_only": _tt,
                                  "p_exact_two_sided": round(p, 5)},
                      "final_epoch_mean_CE": round(ce, 4)})
        print(f"{K:>6}{len(pool):>9}{len(pool)/max(1,ncls):>10.1f}{len(ev):>8}"
              f"{r:>8.3f}{t:>8.3f}{t-r:>+8.3f}{sh:>8.3f}{p:>9.4f}{ce:>10.4f}", flush=True)

    RESULTS["factual_recall_classcount"] = {
        "_design": ("item budget held fixed at N_FIX across every K, so only the number of distinct "
                    "answer classes varies. Recon is scored on each condition's own eval subset, so "
                    "delta = task - recon is the comparable quantity, not raw conferral."),
        "_prediction": ("under 'too many classes to learn a consistent encoding', delta should be "
                        "positive at small K and fall as K grows. A flat delta near zero says class "
                        "count is not the binding constraint."),
        "n_fix": N_FIX, "epochs": K_EPOCHS, "rank": FACT_RANK, "lr": FACT_LR,
        "bin_distinct_answers": len(_bin_cnt), "sweep": SWEEP,
    }
    print("\nEXP27e:", json.dumps(RESULTS["factual_recall_classcount"], indent=2))



EXP27e  train: 24000 items / 7252 classes
        bin:   570 items / 459 distinct answers (top answer covers 17/570)
        fixed item budget N_FIX = 4183 (smallest pool across K=[50, 200, 800, 3200]); every condition subsampled to this

     K  train n  ex/class  eval n   recon    task   delta    shuf        p  final CE
    50     4183      83.7     108   0.454   0.667  +0.213   0.037   0.0006    0.5101
   200     4183      20.9     183   0.443   0.437  -0.005   0.016   1.0000    0.7776
   800     4183       5.4     285   0.372   0.425  +0.053   0.004   0.0674    0.9864
  3200     4183       2.2     407   0.354   0.391  +0.037   0.005   0.0912    1.2292

EXP27e: {
  "_design": "item budget held fixed at N_FIX across every K, so only the number of distinct answer classes varies. Recon is scored on each condition's own eval subset, so delta = task - recon is the comparable quantity, not raw conferral.",
  "_prediction": "under 'too many classes to learn a consistent encoding', delta sh

In [12]:
# EXP27f — class-matched training + class-matched sweep
if not globals().get("RUN_CLASSMATCH", True):
    print("EXP27f skipped (RUN_CLASSMATCH = False)")
else:
    # === EXP27f: CLASS-MATCHED TRAINING — do the EVAL'S OWN answer classes help? ===
    #
    # THE IDEA: separate the CLASSES from the QUESTIONS. Train only on items whose ANSWER is one of
    # the bin's answer classes, using DIFFERENT questions. Train and eval are disjoint by prompt
    # (CELL D dedups on prompt, then the split partitions items), so no arm can memorize a test
    # prompt. Any gain in arm A is therefore a learned class encoding, not memorization.
    #
    # THE CATCH, measured before anything trains. The bin selects on rarity -- facts the donor knows
    # and the recipient does not -- so many bin answers appear exactly ONCE in the whole fact set,
    # which means they land in eval and have NO training item sharing that answer. Those bin items
    # are untestable by this design, not by accident but by construction. So we compute COVERAGE
    # first and run the comparison on BIN_COVERED: the bin items whose answer class does occur in
    # training. That subset is where the question is even askable. The full bin is reported too, but
    # it dilutes arm A with items it could not have learned anything about.
    #
    #   arm A  train items whose answer is one of the COVERED bin classes
    #   arm B  same item count, drawn from ALL classes -> isolates class-focus from data volume
    #   arm C  same item count, frequency-matched classes NOT in the bin -> isolates THESE classes
    #          from any narrow class set. Biased DOWNWARD (CE over non-bin classes pushes mass away
    #          from the bin's answers), so read C as a floor, not a fair midpoint.
    #
    # READ:  A > B  => matching the eval's classes helps beyond data volume (the hypothesis)
    #        A ~ B  => the sweep's K=50 effect is class COUNT, not class IDENTITY
    for _n in ["FX9t","FX9e","FWr","FWr_d","Fmu9d","Fmu2","Fmu2d","FT_TRAIN","FT_EVAL",
               "F_UNSOLV","F_first","_mcnemar","FACT_RANK","FACT_LR","FACT_WD","F_task_maps"]:
        assert _n in globals(), f"{_n} missing -- run EXP27b first, same kernel"

    from collections import Counter as _C
    from bisect import bisect_left as _bl
    CM_EPOCHS  = globals().get("CM_EPOCHS", 8)
    CM_MIN_POOL = globals().get("CM_MIN_POOL", 200)   # below this, arm A cannot be trained
    CM_MIN_EVAL = globals().get("CM_MIN_EVAL", 40)    # below this, the covered subset is unreadable

    _trcnt      = _C(p["tok"] for p in FT_TRAIN)
    BIN_CLASSES = {FT_EVAL[j]["tok"] for j in F_UNSOLV}
    COVERED     = {c for c in BIN_CLASSES if _trcnt[c] > 0}
    BIN_COVERED = [j for j in F_UNSOLV if FT_EVAL[j]["tok"] in COVERED]
    pool_A      = [i for i, p in enumerate(FT_TRAIN) if p["tok"] in COVERED]
    NCM         = len(pool_A)
    _sup        = sorted(_trcnt[c] for c in COVERED)

    print(f"EXP27f  bin: {len(F_UNSOLV)} items over {len(BIN_CLASSES)} answer classes")
    print(f"        COVERAGE: {len(COVERED)}/{len(BIN_CLASSES)} bin classes occur in training "
          f"({len(COVERED)/max(1,len(BIN_CLASSES)):.0%})")
    print(f"        -> {len(BIN_COVERED)}/{len(F_UNSOLV)} bin items are testable by this design")
    print(f"        arm A pool: {NCM} train items"
          + (f"  (median {_sup[len(_sup)//2]} train examples per covered class)" if _sup else ""))

    CM = {"_design": ("arms train on different QUESTIONS with the same ANSWER classes; train/eval "
                      "are prompt-disjoint so no arm can memorize a test prompt. Primary metric is "
                      "BIN_COVERED -- bin items whose answer class occurs in training at all."),
          "n_bin": len(F_UNSOLV), "n_bin_classes": len(BIN_CLASSES),
          "n_covered_classes": len(COVERED), "n_bin_covered": len(BIN_COVERED),
          "coverage_frac": round(len(COVERED)/max(1,len(BIN_CLASSES)), 4),
          "arm_pool_items": NCM}

    if NCM < CM_MIN_POOL or len(BIN_COVERED) < CM_MIN_EVAL:
        CM["_status"] = (f"NOT RUNNABLE: arm A pool {NCM} (need {CM_MIN_POOL}), covered bin items "
                         f"{len(BIN_COVERED)} (need {CM_MIN_EVAL}). This is a property of the task, "
                         f"not a bug: the unsolvable bin selects for answers so rare they occur once "
                         f"in the corpus, so there is no second question with the same answer to "
                         f"train on. Report it as a limit on the design, and lean on the class-count "
                         f"sweep instead.")
        print("\n  >>> " + CM["_status"])
    else:
        _r = random.Random(5)
        pool_B = _r.sample(range(len(FT_TRAIN)), NCM)
        _av = sorted([c for c in _trcnt if c not in BIN_CLASSES], key=lambda c: _trcnt[c])
        _ac = [_trcnt[c] for c in _av]; _used = [False]*len(_av); _match = []
        for c in sorted(COVERED, key=lambda x: -_trcnt[x]):
            t = _trcnt[c]; i = _bl(_ac, t); best = bd = None
            for j in range(max(0, i-60), min(len(_av), i+61)):
                if _used[j]: continue
                d = abs(_ac[j]-t)
                if bd is None or d < bd: best, bd = j, d
            if best is None: best = next((j for j in range(len(_av)) if not _used[j]), None)
            if best is not None: _used[best] = True; _match.append(_av[best])
        _ms = set(_match)
        _cAll = [i for i, p in enumerate(FT_TRAIN) if p["tok"] in _ms]
        pool_C = _r.sample(_cAll, min(NCM, len(_cAll)))
        print(f"        arm C pool: {len(pool_C)} items over {len(_ms)} frequency-matched non-bin classes\n")

        def _fit(idx_list, tag):
            torch.manual_seed(0)
            d9, d2 = FWr.shape
            b = Fmu2.clone().to(DEVICE).float().requires_grad_(True)
            A = (torch.randn(d9, FACT_RANK, device=DEVICE)/math.sqrt(d9)).requires_grad_(True)
            B = torch.zeros(FACT_RANK, d2, device=DEVICE).requires_grad_(True)
            opt = torch.optim.Adam([{"params":[A,B],"weight_decay":FACT_WD},
                                    {"params":[b],"weight_decay":0.0}], lr=FACT_LR)
            model_2b.requires_grad_(False)
            h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
            try:
                idx, ce = list(idx_list), float("nan")
                for ep in range(CM_EPOCHS):
                    random.Random(900+ep).shuffle(idx); tot = n = 0
                    for s in range(0, len(idx), ARITH_BATCH):
                        sub = idx[s:s+ARITH_BATCH]
                        xc = FX9t[sub].to(DEVICE) - Fmu9d
                        _graft["vec"] = xc @ FWr_d + (xc @ A) @ B + b
                        ids, m = left_pad([FT_TRAIN[k]["ids"] for k in sub], tokenizer.pad_token_id)
                        lg = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:,-1,:].float()
                        tgt = torch.tensor([FT_TRAIN[k]["tok"] for k in sub], device=DEVICE)
                        loss = F.cross_entropy(lg, tgt)
                        opt.zero_grad(); loss.backward(); opt.step()
                        tot += float(loss.item()); n += 1
                    ce = tot/max(1,n)
                print(f"      {tag}: {len(idx)} items, final mean CE {ce:.4f}", flush=True)
            finally:
                h.remove(); _graft["vec"] = None; model_2b.requires_grad_(True)
            return (FWr_d + A @ B).detach().clone(), b.detach().clone(), ce

        def _ev(W, b, idxs): return F_first(lambda s: (FX9e[s].to(DEVICE)-Fmu9d) @ W + b, idxs)
        _mA = _fit(pool_A, "arm A (bin's own classes)")
        _mB = _fit(pool_B, "arm B (diffuse, matched volume)")
        _mC = _fit(pool_C, "arm C (freq-matched non-bin)")
        _Wg, _bg = F_task_maps[0]
        for lab, idxs in [("covered", BIN_COVERED), ("fullbin", F_UNSOLV)]:
            a, b_, c_ = _ev(*_mA[:2], idxs), _ev(*_mB[:2], idxs), _ev(*_mC[:2], idxs)
            g = _ev(_Wg, _bg, idxs)
            r = F_first(lambda s: (FX9e[s].to(DEVICE)-Fmu9d) @ FWr_d + Fmu2d, idxs)
            CM[f"{lab}_n"] = len(idxs)
            CM[f"{lab}_recon_first"]        = fmt(wilson_bools(r))
            CM[f"{lab}_generaltrain_first"] = fmt(wilson_bools(g))
            CM[f"{lab}_armA_binclasses"]    = fmt(wilson_bools(a))
            CM[f"{lab}_armB_diffuse"]       = fmt(wilson_bools(b_))
            CM[f"{lab}_armC_nonbin"]        = fmt(wilson_bools(c_))
            CM[f"{lab}_mcnemar_A_vs_B"] = (lambda z: {"B_only":z[0],"A_only":z[1],"p":round(z[2],5)})(_mcnemar(b_, a))
            CM[f"{lab}_mcnemar_A_vs_recon"] = (lambda z: {"recon_only":z[0],"A_only":z[1],"p":round(z[2],5)})(_mcnemar(r, a))
        CM["final_CE"] = {"A": round(_mA[2],4), "B": round(_mB[2],4), "C": round(_mC[2],4)}
        CM["_arm_C_is_a_floor"] = ("CE over non-bin classes pushes mass away from the bin's answers, "
                                   "so C understates. Treat it as a floor, not a fair midpoint.")
        CM["_primary"] = "read the covered_* rows; fullbin_* dilutes arm A with untestable items"
        # ---------------------------------------------------------------------------------
        # PART 2: CLASS-MATCHED SWEEP. Part 1 trains on ALL covered bin classes at once. But
        # learning to encode the bin's answers only helps if there are few enough of them to
        # encode consistently -- so sweep K over the BIN'S OWN classes and watch task - recon.
        # This is not EXP27e: that sweeps the top-K classes of the whole corpus and tests on
        # whatever bin items fall inside them. Here every class is already a bin class, so K
        # varies the count while class identity is held matched throughout.
        # Recon is re-scored on each K's own eval subset, so task - recon is the comparable
        # quantity across K; the shuffled arm catches the restricted-output prior at low K.
        CM_K_SWEEP     = globals().get("CM_K_SWEEP", [50, 200, 800, "all"])
        CM_SWEEP_MINEV = globals().get("CM_SWEEP_MINEV", 30)
        CM_SWEEP_MINIT = globals().get("CM_SWEEP_MINIT", 150)

        _ranked = [c for c, _ in sorted(((c, _trcnt[c]) for c in COVERED), key=lambda x: -x[1])]
        _cand = {}
        for K in CM_K_SWEEP:
            kk = len(_ranked) if K == "all" else min(int(K), len(_ranked))
            keep = set(_ranked[:kk])
            ev_k = [j for j in F_UNSOLV if FT_EVAL[j]["tok"] in keep]
            pl_k = [i for i, p in enumerate(FT_TRAIN) if p["tok"] in keep]
            _cand[K] = (kk, keep, ev_k, pl_k)

        _viable = {K: v for K, v in _cand.items() if len(v[2]) >= CM_SWEEP_MINEV and len(v[3]) >= 32}
        _nfix = min((len(v[3]) for v in _viable.values()), default=0)
        _fixed = _nfix >= CM_SWEEP_MINIT
        print(f"\n  PART 2: class-matched sweep over the bin's own {len(_ranked)} covered classes")
        if _fixed:
            print(f"          item budget FIXED at {_nfix} across K, so only class count varies")
        else:
            print(f"          item budget NOT fixed (smallest pool {_nfix} < {CM_SWEEP_MINIT}): "
                  f"train volume covaries with K, so read this against EXP27e, which does fix it")

        SW2 = []
        print(f"\n{'K':>6}{'classes':>9}{'train n':>9}{'ex/cls':>8}{'eval n':>8}"
              f"{'recon':>8}{'task':>8}{'delta':>8}{'shuf':>8}{'p':>9}")
        for K in CM_K_SWEEP:
            kk, keep, ev_k, pl_k = _cand[K]
            if K not in _viable:
                print(f"{str(K):>6}{kk:>9}{len(pl_k):>9}{'-':>8}{len(ev_k):>8}   SKIPPED (too small)")
                continue
            pool = list(pl_k)
            if _fixed:
                random.Random(77).shuffle(pool); pool = pool[:_nfix]
            ncl = len({FT_TRAIN[i]["tok"] for i in pool})
            W_k, b_k, ce_k = _fit(pool, f"sweep K={K}")
            t_ok = _ev(W_k, b_k, ev_k)
            r_ok = F_first(lambda s: (FX9e[s].to(DEVICE) - Fmu9d) @ FWr_d + Fmu2d, ev_k)
            _rg = random.Random(41); _pp = list(ev_k)
            for _ in range(200):
                _rg.shuffle(_pp)
                if all(FT_EVAL[_pp[i]]["tok"] != FT_EVAL[ev_k[i]]["tok"] for i in range(len(ev_k))): break
            _PM = dict(zip(ev_k, _pp))
            s_ok = F_first(lambda s: (FX9e[[_PM[j] for j in s]].to(DEVICE) - Fmu9d) @ W_k + b_k, ev_k)
            t, r, sh = (sum(x) / len(ev_k) for x in (t_ok, r_ok, s_ok))
            _ro, _to, p = _mcnemar(r_ok, t_ok)
            SW2.append({"K": K, "n_classes": ncl, "train_items": len(pool),
                        "examples_per_class": round(len(pool) / max(1, ncl), 2),
                        "eval_n": len(ev_k), "recon_first": round(r, 4), "task_first": round(t, 4),
                        "delta_task_minus_recon": round(t - r, 4),
                        "task_SHUFFLED_first": round(sh, 4),
                        "mcnemar": {"recon_only": _ro, "task_only": _to, "p_exact_two_sided": round(p, 5)},
                        "final_CE": round(ce_k, 4)})
            print(f"{str(K):>6}{ncl:>9}{len(pool):>9}{len(pool)/max(1,ncl):>8.1f}{len(ev_k):>8}"
                  f"{r:>8.3f}{t:>8.3f}{t-r:>+8.3f}{sh:>8.3f}{p:>9.4f}", flush=True)

        CM["classmatched_sweep"] = SW2
        CM["_sweep_design"] = ("K ranges over the BIN'S OWN covered classes (top-K by train "
                               "frequency), so class identity is matched at every K and only the "
                               "count varies. Recon is re-scored per subset; the shuffled arm "
                               "catches a restricted-output prior at low K."
                               + (f" Item budget fixed at {_nfix}." if _fixed else
                                  " Item budget NOT fixed -- volume covaries with K."))
        CM["_sweep_budget_fixed"] = bool(_fixed)


    RESULTS["factual_recall_classmatch"] = CM
    print("\nEXP27f:", json.dumps(CM, indent=2))



EXP27f  bin: 570 items over 459 answer classes
        COVERAGE: 376/459 bin classes occur in training (82%)
        -> 486/570 bin items are testable by this design
        arm A pool: 6041 train items  (median 5 train examples per covered class)
        arm C pool: 3650 items over 376 frequency-matched non-bin classes

      arm A (bin's own classes): 6041 items, final mean CE 0.8515
      arm B (diffuse, matched volume): 6041 items, final mean CE 1.4137
      arm C (freq-matched non-bin): 3650 items, final mean CE 0.6779

  PART 2: class-matched sweep over the bin's own 376 covered classes
          item budget FIXED at 3963 across K, so only class count varies

     K  classes  train n  ex/cls  eval n   recon    task   delta    shuf        p
      sweep K=50: 3963 items, final mean CE 0.4767
    50       50     3963    79.3     125   0.448   0.608  +0.160   0.072   0.0037
      sweep K=200: 3963 items, final mean CE 0.7662
   200      200     3963    19.8     306   0.359   0.415  +

In [13]:
# EXP27g — properly-pooled oracle
if not globals().get("RUN_POOLED_ORACLE", True):
    print("EXP27g — properly-pooled oracle skipped (RUN_POOLED_ORACLE = False)")
else:
    # === EXP27g: PROPERLY-POOLED ORACLE — apples-to-apples with the arithmetic EXP9 oracle ===
    #
    # WHY THE EARLIER FACTS ORACLE WAS VOID. It trained on the 570-item bin alone. Donor states are
    # 3,584-dimensional, so 570 of them are linearly INDEPENDENT: a linear map can send each one
    # anywhere, independently. Combined with EXP14 (per-item free vectors reach 100%), a score of
    # 1.000 was guaranteed by construction -- shuffle the gold labels and it still hits 1.000. That
    # number measures parameter count, not donor content. Discard it.
    #
    # WHAT EXP9 ACTUALLY DID (Gemma_eval_EXP13-21, cell 20):
    #     pool = train + evalp          # 3000 + 2000 = 5000 items, well above 3,584
    #     tgt  = first answer token; full-rank map; recon init; lr 1e-3; 2x epochs
    # Above the dimension count the states are linearly DEPENDENT, so the map cannot treat items
    # separately and must find shared structure. That is why arithmetic's 0.119 means something.
    #
    # THIS CELL reproduces that construction on facts: pool = FT_TRAIN + FT_EVAL (27,000 >> 3,584),
    # first-token CE, recon init, full rank at lr 1e-3 to match EXP9, and also the facts-tuned
    # rank-32 config so a poor full-rank result cannot be blamed on the schedule. Best-checkpoint is
    # tracked ON THE BIN, which is legitimate here and only for the oracle: it is an upper bound by
    # construction, and reporting its best is what makes it a ceiling.
    for _n in ["FX9t","FX9e","FWr","FWr_d","Fmu9d","Fmu2","Fmu2d","FT_TRAIN","FT_EVAL",
               "F_UNSOLV","F_first","F_full","_mcnemar","MON","F_task_maps"]:
        assert _n in globals(), f"{_n} missing -- run EXP27b first, same kernel"

    O2_EPOCHS    = globals().get("O2_EPOCHS", 8)
    O2_MAX_STEPS = globals().get("O2_MAX_STEPS", 6000)
    O2_MON       = globals().get("O2_MON", 500)
    O2_ARMS      = globals().get("O2_ARMS", [("fullrank_lr1e-3", 0, 1e-3), ("rank32_lr1e-4", 32, 1e-4)])

    P_ITEMS = FT_TRAIN + FT_EVAL
    P_X9    = torch.cat([FX9t, FX9e], 0)
    print(f"EXP27g  pooled oracle: {len(P_ITEMS)} items ({len(FT_TRAIN)} train + {len(FT_EVAL)} eval) "
          f"vs donor dim {FWr.shape[0]}")
    print(f"        {len(P_ITEMS)/FWr.shape[0]:.1f}x the donor dimension -> memorization is ruled out\n")

    def _fit_pooled(rank, lr, tag):
        torch.manual_seed(0)
        d9, d2 = FWr.shape
        b = Fmu2.clone().to(DEVICE).float().requires_grad_(True)
        if rank:
            A = (torch.randn(d9, rank, device=DEVICE) / math.sqrt(d9)).requires_grad_(True)
            Bm = torch.zeros(rank, d2, device=DEVICE).requires_grad_(True)
            groups = [{"params": [A, Bm], "weight_decay": FACT_WD}, {"params": [b], "weight_decay": 0.0}]
            def W_eff(): return FWr_d + A @ Bm
            def vec(x):
                xc = x - Fmu9d
                return xc @ FWr_d + (xc @ A) @ Bm + b
        else:
            Wf = FWr.clone().to(DEVICE).requires_grad_(True)
            groups = [{"params": [Wf, b], "weight_decay": 0.0}]
            def W_eff(): return Wf
            def vec(x): return (x - Fmu9d) @ Wf + b
        opt = torch.optim.Adam(groups, lr=lr)
        model_2b.requires_grad_(False)
        h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
        try:
            idx = list(range(len(P_ITEMS))); g = 0; stop = False
            bW, bb = W_eff().detach().clone(), b.detach().clone()
            bacc = sum(F_first(lambda s: (FX9e[s].to(DEVICE) - Fmu9d) @ bW + bb, MON)) / len(MON)
            bstep = 0
            print(f"      {tag} step     0 (recon init)  bin = {bacc:.3f}", flush=True)
            for ep in range(O2_EPOCHS):
                if stop: break
                random.Random(4000 + ep).shuffle(idx)
                for s in range(0, len(idx), ARITH_BATCH):
                    sub = idx[s:s + ARITH_BATCH]
                    _graft["vec"] = vec(P_X9[sub].to(DEVICE))
                    ids, m = left_pad([P_ITEMS[k]["ids"] for k in sub], tokenizer.pad_token_id)
                    lg = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                    tgt = torch.tensor([P_ITEMS[k]["tok"] for k in sub], device=DEVICE)
                    loss = F.cross_entropy(lg, tgt)
                    opt.zero_grad(); loss.backward(); opt.step(); g += 1
                    if g % O2_MON == 0:
                        _W, _b = W_eff().detach().clone(), b.detach().clone()
                        a = sum(F_first(lambda s: (FX9e[s].to(DEVICE) - Fmu9d) @ _W + _b, MON)) / len(MON)
                        fl = ""
                        if a > bacc: bacc, bW, bb, bstep, fl = a, _W, _b, g, "  <- best"
                        print(f"      {tag} step {g:5d} (ep {ep})  CE = {float(loss.item()):.4f}"
                              f"   bin = {a:.3f}{fl}", flush=True)
                    if g >= O2_MAX_STEPS: stop = True; break
            print(f"      {tag} BEST bin = {bacc:.3f} at step {bstep}"
                  f"{'  (never beat the recon init)' if bstep == 0 else ''}", flush=True)
        finally:
            h.remove(); _graft["vec"] = None; model_2b.requires_grad_(True)
        return bW, bb, bstep

    OUT = {
        "_design": ("pool = FT_TRAIN + FT_EVAL, matching EXP9's `pool = train + evalp`. Pool size is "
                    "far above the donor dimension, so the map cannot assign items independently and "
                    "must find shared structure. This is what makes the number comparable to "
                    "arithmetic's oracle; the 570-item version was not."),
        "_supersedes": ("the earlier bin-only facts oracle (oracle_first 1.000) is void: 570 < 3584 "
                        "means linearly independent donor states, so a perfect score was guaranteed "
                        "regardless of donor content. Do not report it."),
        "pool_items": len(P_ITEMS), "donor_dim": int(FWr.shape[0]),
        "pool_over_dim": round(len(P_ITEMS) / FWr.shape[0], 2),
        "epochs": O2_EPOCHS, "max_steps": O2_MAX_STEPS,
        "recon_first": fmt(wilson_bools(F_first(lambda s: (FX9e[s].to(DEVICE)-Fmu9d) @ FWr_d + Fmu2d, F_UNSOLV))),
        "task_first":  fmt(wilson_bools(F_first(lambda s: (FX9e[s].to(DEVICE)-Fmu9d) @ F_task_maps[0][0] + F_task_maps[0][1], F_UNSOLV))),
    }
    for tag, rank, lr in O2_ARMS:
        W, b, bstep = _fit_pooled(rank, lr, tag)
        ok  = F_first(lambda s: (FX9e[s].to(DEVICE) - Fmu9d) @ W + b, F_UNSOLV)
        okf = F_full(lambda s: (FX9e[s].to(DEVICE) - Fmu9d) @ W + b, F_UNSOLV)
        tk  = F_first(lambda s: (FX9e[s].to(DEVICE) - Fmu9d) @ F_task_maps[0][0] + F_task_maps[0][1], F_UNSOLV)
        OUT[f"oracle_{tag}_first"] = fmt(wilson_bools(ok))
        OUT[f"oracle_{tag}_full"]  = fmt(wilson_bools(okf))
        OUT[f"oracle_{tag}_best_step"] = bstep
        OUT[f"mcnemar_{tag}_vs_task_first"] = (lambda z: {"task_only": z[0], "oracle_only": z[1],
                                                          "p": round(z[2], 5)})(_mcnemar(tk, ok))
        # Arithmetic reference, measured: RESULTS["oracle_stitch"] from the EXP13-21 run.
    # oracle_unsolv_first 0.980, task_unsolv_first_pooled 0.886, oracle shuffle 0.118 (so the
    # arithmetic oracle is content-driven, not a prior). Recovery = task / oracle.
    ARITH_TASK_FIRST, ARITH_ORACLE_FIRST = 0.886, 0.980
    OUT["_arithmetic_reference"] = {
        "task_first": ARITH_TASK_FIRST, "oracle_first": ARITH_ORACLE_FIRST,
        "oracle_shuffle_first": 0.118,
        "recovery_task_over_oracle": round(ARITH_TASK_FIRST / ARITH_ORACLE_FIRST, 4),
        "note": "learned map recovers ~90% of the privileged map on arithmetic"}
    _tf = float(OUT["task_first"].split()[0])
    for tag, _, _ in O2_ARMS:
        _o = float(OUT[f"oracle_{tag}_first"].split()[0])
        OUT[f"recovery_{tag}"] = round(_tf / _o, 4) if _o > 0 else None
    print("\n      RECOVERY (task / oracle), the headline comparison:")
    print(f"        arithmetic          {ARITH_TASK_FIRST:.3f} / {ARITH_ORACLE_FIRST:.3f} "
          f"= {ARITH_TASK_FIRST/ARITH_ORACLE_FIRST:.1%}")
    for tag, _, _ in O2_ARMS:
        _o = float(OUT[f"oracle_{tag}_first"].split()[0])
        print(f"        facts {tag:<16}{_tf:.3f} / {_o:.3f} = "
              f"{(_tf/_o if _o else float('nan')):.1%}")
    print("      A facts recovery well BELOW ~90% means the content is there and the ordinary")
    print("      map fails to LEARN it. Near 90% means no shared linear map does better.")
    RESULTS["factual_recall_oracle_pooled"] = OUT
    print("\nEXP27g:", json.dumps(OUT, indent=2))
    print("\nREAD: compare oracle_*_first against task_first. A large gap with a pool this size means")
    print("      the structure is there and the ordinary map fails to LEARN it. A small gap means no")
    print("      shared linear map does better -- the arithmetic conclusion.")



EXP27g  pooled oracle: 27000 items (24000 train + 3000 eval) vs donor dim 3584
        7.5x the donor dimension -> memorization is ruled out

      fullrank_lr1e-3 step     0 (recon init)  bin = 0.287
      fullrank_lr1e-3 step   500 (ep 0)  CE = 3.7612   bin = 0.338  <- best
      fullrank_lr1e-3 step  1000 (ep 0)  CE = 4.9204   bin = 0.531  <- best
      fullrank_lr1e-3 step  1500 (ep 0)  CE = 4.8617   bin = 0.662  <- best
      fullrank_lr1e-3 step  2000 (ep 1)  CE = 1.1504   bin = 0.725  <- best
      fullrank_lr1e-3 step  2500 (ep 1)  CE = 0.9905   bin = 0.762  <- best
      fullrank_lr1e-3 step  3000 (ep 1)  CE = 1.0404   bin = 0.762
      fullrank_lr1e-3 step  3500 (ep 2)  CE = 0.8353   bin = 0.806  <- best
      fullrank_lr1e-3 step  4000 (ep 2)  CE = 0.6553   bin = 0.794
      fullrank_lr1e-3 step  4500 (ep 2)  CE = 1.9250   bin = 0.831  <- best
      fullrank_lr1e-3 step  5000 (ep 2)  CE = 1.0955   bin = 0.831
      fullrank_lr1e-3 step  5500 (ep 3)  CE = 0.9621   bin = 0.844

TypeError: 'list' object is not callable

In [14]:
# === EXP27g SALVAGE — run this INSTEAD of re-running EXP27g. No retraining of the fullrank arm.
#
# WHAT BROKE. EXP27f bound `_match = []` as a local list, clobbering the `_match(gen_text, gold)`
# answer-matcher defined in EXP27b -- notebook cells share one global namespace. EXP27g's F_full
# then called a list. My naming bug, not a problem with your run.
#
# WHAT SURVIVED. The fullrank arm finished (BEST bin 0.850 at step 6000) and its W, b are still
# bound as globals, because the exception fired after _fit_pooled returned. So this restores
# _match, scores the fullrank arm from the weights already in memory, then trains only the
# rank32 arm. You save the 6000-step fullrank run.

# ---- 1. restore the clobbered matcher, verbatim from EXP27b ----
def _norm(t):
    return "".join(c for c in t.lower() if c.isascii() and (c.isalnum() or c == " ")).strip()

def _match(gen_text, gold):
    """Strict: normalized equality, or gold followed by a word boundary."""
    g, p = _norm(gold), _norm(gen_text.split("\n")[0])
    return bool(g) and (p == g or p.startswith(g + " "))

assert _match("Paris\nsomething", "Paris") and not _match("Australia", "Au"), "matcher restored wrong"
print("_match restored and self-tested")

# ---- 2. score the fullrank arm from the weights still in memory ----
_have = all(n in globals() for n in ("W", "b", "OUT", "O2_ARMS", "P_ITEMS"))
assert _have, ("EXP27g's globals are gone -- the kernel was restarted. Re-run the EXP27g cell "
               "from the notebook instead; the _match bug is fixed there.")
_fulltag = O2_ARMS[0][0]
print(f"scoring {_fulltag} from the weights already in memory (no retrain)...")
_Wf, _bf = W.detach().clone(), b.detach().clone()

def _mk(Wx, bx): return lambda s: (FX9e[s].to(DEVICE) - Fmu9d) @ Wx + bx
_tk = F_first(_mk(F_task_maps[0][0], F_task_maps[0][1]), F_UNSOLV)
for tag, Wx, bx, bstep in [(_fulltag, _Wf, _bf, globals().get("bstep", None))]:
    ok, okf = F_first(_mk(Wx, bx), F_UNSOLV), F_full(_mk(Wx, bx), F_UNSOLV)
    OUT[f"oracle_{tag}_first"] = fmt(wilson_bools(ok))
    OUT[f"oracle_{tag}_full"]  = fmt(wilson_bools(okf))
    OUT[f"oracle_{tag}_best_step"] = bstep
    OUT[f"mcnemar_{tag}_vs_task_first"] = (lambda z: {"task_only": z[0], "oracle_only": z[1],
                                                      "p": round(z[2], 5)})(_mcnemar(_tk, ok))
    print(f"  {tag}: first {OUT[f'oracle_{tag}_first']}  full {OUT[f'oracle_{tag}_full']}")
OUT["_fullrank_hit_step_cap"] = ("the fullrank arm was still improving when O2_MAX_STEPS ended "
                                 "it, so its score is a LOWER bound on the ceiling -- which only "
                                 "widens the gap to the task map")

# ---- 3. train the remaining arm(s) ----
for tag, rank, lr in O2_ARMS[1:]:
    Wx, bx, bstep = _fit_pooled(rank, lr, tag)
    ok, okf = F_first(_mk(Wx, bx), F_UNSOLV), F_full(_mk(Wx, bx), F_UNSOLV)
    OUT[f"oracle_{tag}_first"] = fmt(wilson_bools(ok))
    OUT[f"oracle_{tag}_full"]  = fmt(wilson_bools(okf))
    OUT[f"oracle_{tag}_best_step"] = bstep
    OUT[f"mcnemar_{tag}_vs_task_first"] = (lambda z: {"task_only": z[0], "oracle_only": z[1],
                                                      "p": round(z[2], 5)})(_mcnemar(_tk, ok))

# ---- 4. recovery comparison ----
ARITH_TASK_FIRST, ARITH_ORACLE_FIRST = 0.886, 0.980
OUT["_arithmetic_reference"] = {
    "task_first": ARITH_TASK_FIRST, "oracle_first": ARITH_ORACLE_FIRST,
    "oracle_shuffle_first": 0.118,
    "recovery_task_over_oracle": round(ARITH_TASK_FIRST / ARITH_ORACLE_FIRST, 4),
    "note": "learned map recovers ~90% of the privileged map on arithmetic"}
_tf = float(OUT["task_first"].split()[0])
print("\n  RECOVERY (task / oracle), the headline comparison:")
print(f"    arithmetic          {ARITH_TASK_FIRST:.3f} / {ARITH_ORACLE_FIRST:.3f} "
      f"= {ARITH_TASK_FIRST/ARITH_ORACLE_FIRST:.1%}")
for tag, _, _ in O2_ARMS:
    if f"oracle_{tag}_first" not in OUT: continue
    _o = float(OUT[f"oracle_{tag}_first"].split()[0])
    OUT[f"recovery_{tag}"] = round(_tf / _o, 4) if _o > 0 else None
    print(f"    facts {tag:<18}{_tf:.3f} / {_o:.3f} = {(_tf/_o if _o else float('nan')):.1%}")
print("  Facts recovery well BELOW ~90% means the content is there and the ordinary map fails")
print("  to LEARN it. Near 90% means no shared linear map does better.")

RESULTS["factual_recall_oracle_pooled"] = OUT
print("\nEXP27g:", json.dumps(OUT, indent=2))


_match restored and self-tested
scoring fullrank_lr1e-3 from the weights already in memory (no retrain)...
  fullrank_lr1e-3: first 0.835 [0.802, 0.863]  full 0.725 [0.686, 0.760]
      rank32_lr1e-4 step     0 (recon init)  bin = 0.287
      rank32_lr1e-4 step   500 (ep 0)  CE = 2.6253   bin = 0.325  <- best
      rank32_lr1e-4 step  1000 (ep 0)  CE = 2.9864   bin = 0.325
      rank32_lr1e-4 step  1500 (ep 0)  CE = 2.7930   bin = 0.362  <- best
      rank32_lr1e-4 step  2000 (ep 1)  CE = 3.1025   bin = 0.344
      rank32_lr1e-4 step  2500 (ep 1)  CE = 2.0232   bin = 0.369  <- best
      rank32_lr1e-4 step  3000 (ep 1)  CE = 2.6457   bin = 0.394  <- best
      rank32_lr1e-4 step  3500 (ep 2)  CE = 2.3874   bin = 0.362
      rank32_lr1e-4 step  4000 (ep 2)  CE = 2.3019   bin = 0.375
      rank32_lr1e-4 step  4500 (ep 2)  CE = 2.2119   bin = 0.400  <- best
      rank32_lr1e-4 step  5000 (ep 2)  CE = 2.0102   bin = 0.388
      rank32_lr1e-4 step  5500 (ep 3)  CE = 1.7118   bin = 0.412  <-

In [15]:
# === EXP27h: UNSEEN-CLASS SPLIT + RANDOM-K SELECTION ===
# Paste after EXP27f, same kernel. Reuses its trained maps if they are still in scope and
# retrains only what is missing, so it is cheap to run right after 27f.
#
# PART 1 -- UNSEEN CLASSES. 27f reported `covered` (486 items) and `fullbin` (570). The
# difference is the 84 bin items whose answer class arm A never trained on. Subtracting the
# reported rates implies arm A scores ~0.048 there against recon's ~0.214 -- FOUR TIMES WORSE
# than a map that never saw a label. If that holds when measured directly it is the strongest
# statement available: the task map is not learning a general donor->answer read-out, because a
# general read-out would transfer to new classes. It anti-transfers, i.e. CE actively suppresses
# mass on classes outside its training set. Chance-level would mean "needs examples per class";
# BELOW the label-free baseline means something stronger.
#
# PART 2 -- RANDOM-K. In 27f's sweep, K classes are taken top-K BY TRAINING FREQUENCY, so class
# count and answer frequency are perfectly correlated and a reviewer can ask which one moves the
# delta. Here the K classes are drawn at RANDOM from the covered set at the same fixed budget.
# If the falling delta reproduces, it is class COUNT. If it flattens, it was frequency.
for _n in ["FX9t","FX9e","FWr","FWr_d","Fmu9d","Fmu2","Fmu2d","FT_TRAIN","FT_EVAL",
           "F_UNSOLV","F_first","F2_top","_mcnemar","FACT_RANK","FACT_LR","FACT_WD","F_task_maps"]:
    assert _n in globals(), f"{_n} missing -- run EXP27b (and EXP27f) first, same kernel"

from collections import Counter as _C
H_EPOCHS   = globals().get("H_EPOCHS", 8)
H_RK_SEEDS = globals().get("H_RK_SEEDS", [0, 1])       # random draws per K, so one lucky sample can't carry it
H_K_SWEEP  = globals().get("H_K_SWEEP", [50, 200, 800])
H_MIN_EV   = globals().get("H_MIN_EV", 30)

_trc  = _C(p["tok"] for p in FT_TRAIN)
_bcls = {FT_EVAL[j]["tok"] for j in F_UNSOLV}
_cov  = {c for c in _bcls if _trc[c] > 0}
BIN_COV    = [j for j in F_UNSOLV if FT_EVAL[j]["tok"] in _cov]
BIN_UNSEEN = [j for j in F_UNSOLV if FT_EVAL[j]["tok"] not in _cov]
print(f"EXP27h  bin {len(F_UNSOLV)} = covered {len(BIN_COV)} + UNSEEN {len(BIN_UNSEEN)}")

def _fit2(idx_list, tag, seed=0, epochs=None):
    torch.manual_seed(seed)
    d9, d2 = FWr.shape
    b = Fmu2.clone().to(DEVICE).float().requires_grad_(True)
    A = (torch.randn(d9, FACT_RANK, device=DEVICE) / math.sqrt(d9)).requires_grad_(True)
    B = torch.zeros(FACT_RANK, d2, device=DEVICE).requires_grad_(True)
    opt = torch.optim.Adam([{"params": [A, B], "weight_decay": FACT_WD},
                            {"params": [b], "weight_decay": 0.0}], lr=FACT_LR)
    model_2b.requires_grad_(False)
    h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    try:
        idx, ce = list(idx_list), float("nan")
        for ep in range(epochs or H_EPOCHS):
            random.Random(900 + 17 * seed + ep).shuffle(idx); tot = n = 0
            for s in range(0, len(idx), ARITH_BATCH):
                sub = idx[s:s + ARITH_BATCH]
                xc = FX9t[sub].to(DEVICE) - Fmu9d
                _graft["vec"] = xc @ FWr_d + (xc @ A) @ B + b
                ids, m = left_pad([FT_TRAIN[k]["ids"] for k in sub], tokenizer.pad_token_id)
                lg = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                tgt = torch.tensor([FT_TRAIN[k]["tok"] for k in sub], device=DEVICE)
                loss = F.cross_entropy(lg, tgt)
                opt.zero_grad(); loss.backward(); opt.step()
                tot += float(loss.item()); n += 1
            ce = tot / max(1, n)
        print(f"      {tag}: {len(idx)} items, final mean CE {ce:.4f}", flush=True)
    finally:
        h.remove(); _graft["vec"] = None; model_2b.requires_grad_(True)
    return (FWr_d + A @ B).detach().clone(), b.detach().clone(), ce

def _E(W, b, idxs): return F_first(lambda s: (FX9e[s].to(DEVICE) - Fmu9d) @ W + b, idxs)
def _R(idxs):       return F_first(lambda s: (FX9e[s].to(DEVICE) - Fmu9d) @ FWr_d + Fmu2d, idxs)
def _N(idxs):       return [F2_top[j] == FT_EVAL[j]["tok"] for j in idxs]

# ---------------- PART 1 ----------------
poolA = [i for i, p in enumerate(FT_TRAIN) if p["tok"] in _cov]
if "_mA" in globals():
    WA, bA = _mA[0], _mA[1]; print("      reusing arm A from EXP27f")
else:
    WA, bA, _ = _fit2(poolA, "arm A (refit)")
if "_mB" in globals():
    WB, bB = _mB[0], _mB[1]; print("      reusing arm B from EXP27f")
else:
    WB, bB, _ = _fit2(random.Random(5).sample(range(len(FT_TRAIN)), len(poolA)), "arm B (refit)")
WG, bG = F_task_maps[0]

H = {"_part1": ("the 84 bin items whose answer class arm A never trained on. A general "
                "donor->answer read-out would transfer to them; scoring BELOW the label-free "
                "recon map means CE is suppressing mass on untrained classes instead."),
     "n_covered": len(BIN_COV), "n_unseen": len(BIN_UNSEEN)}
print(f"\n{'subset':>10}{'n':>6}{'native':>9}{'recon':>9}{'general':>9}{'armA':>9}{'armB':>9}")
for lab, idxs in [("covered", BIN_COV), ("UNSEEN", BIN_UNSEEN), ("fullbin", F_UNSOLV)]:
    if not idxs: continue
    nv, rc, gn, aa, bb = _N(idxs), _R(idxs), _E(WG, bG, idxs), _E(WA, bA, idxs), _E(WB, bB, idxs)
    for k, v in [("native", nv), ("recon", rc), ("generaltrain", gn), ("armA", aa), ("armB", bb)]:
        H[f"{lab}_{k}"] = fmt(wilson_bools(v))
    if lab == "UNSEEN":
        H["unseen_mcnemar_armA_vs_recon"] = (lambda z: {"recon_only": z[0], "armA_only": z[1],
            "p_exact_two_sided": round(z[2], 5)})(_mcnemar(rc, aa))
        H["_unseen_read"] = ("armA below recon here = anti-transfer: the task map suppresses "
                             "classes it did not train on, so it learned per-class encodings "
                             "rather than a general read-out.")
    print(f"{lab:>10}{len(idxs):>6}" + "".join(f"{sum(x)/len(idxs):>9.3f}" for x in (nv, rc, gn, aa, bb)), flush=True)

# ---------------- PART 2 ----------------
_covl = sorted(_cov)
_cand = {}
for K in H_K_SWEEP:
    for sd in H_RK_SEEDS:
        kk = min(int(K), len(_covl))
        keep = set(random.Random(1000 * sd + K).sample(_covl, kk))
        ev = [j for j in F_UNSOLV if FT_EVAL[j]["tok"] in keep]
        pl = [i for i, p in enumerate(FT_TRAIN) if p["tok"] in keep]
        _cand[(K, sd)] = (kk, ev, pl)
_ok = {k: v for k, v in _cand.items() if len(v[1]) >= H_MIN_EV and len(v[2]) >= 32}
_nfix = min((len(v[2]) for v in _ok.values()), default=0)
print(f"\n  PART 2: RANDOM-K selection (vs 27f's top-K-by-frequency), budget fixed at {_nfix}")
print(f"{'K':>6}{'seed':>6}{'classes':>9}{'train n':>9}{'ex/cls':>8}{'eval n':>8}"
      f"{'recon':>8}{'task':>8}{'delta':>8}{'shuf':>8}{'p':>9}")
RK = []
for (K, sd), (kk, ev, pl) in sorted(_cand.items()):
    if (K, sd) not in _ok:
        print(f"{K:>6}{sd:>6}{kk:>9}{len(pl):>9}{'-':>8}{len(ev):>8}   SKIPPED (too small)"); continue
    pool = list(pl); random.Random(77).shuffle(pool); pool = pool[:_nfix]
    ncl = len({FT_TRAIN[i]["tok"] for i in pool})
    W, b, ce = _fit2(pool, f"randK K={K} seed={sd}", seed=sd)
    t_ok, r_ok = _E(W, b, ev), _R(ev)
    _rg = random.Random(41); _pp = list(ev)
    for _ in range(200):
        _rg.shuffle(_pp)
        if all(FT_EVAL[_pp[i]]["tok"] != FT_EVAL[ev[i]]["tok"] for i in range(len(ev))): break
    _PM = dict(zip(ev, _pp))
    s_ok = F_first(lambda s: (FX9e[[_PM[j] for j in s]].to(DEVICE) - Fmu9d) @ W + b, ev)
    t, r, sh = (sum(x) / len(ev) for x in (t_ok, r_ok, s_ok))
    _ro, _to, p = _mcnemar(r_ok, t_ok)
    RK.append({"K": K, "seed": sd, "n_classes": ncl, "train_items": len(pool),
               "examples_per_class": round(len(pool) / max(1, ncl), 2), "eval_n": len(ev),
               "recon_first": round(r, 4), "task_first": round(t, 4),
               "delta_task_minus_recon": round(t - r, 4), "task_SHUFFLED_first": round(sh, 4),
               "mcnemar": {"recon_only": _ro, "task_only": _to, "p_exact_two_sided": round(p, 5)},
               "final_CE": round(ce, 4)})
    print(f"{K:>6}{sd:>6}{ncl:>9}{len(pool):>9}{len(pool)/max(1,ncl):>8.1f}{len(ev):>8}"
          f"{r:>8.3f}{t:>8.3f}{t-r:>+8.3f}{sh:>8.3f}{p:>9.4f}", flush=True)

H["randomK_sweep"] = RK
H["randomK_budget"] = _nfix
H["_part2"] = ("K classes drawn at RANDOM from the covered set, fixed budget. 27f drew top-K by "
               "training frequency, which correlates count with answer frequency. If the falling "
               "delta reproduces here it is class COUNT; if it flattens it was frequency.")
_prev = RESULTS.get("factual_recall_classmatch", {}).get("classmatched_sweep")
if _prev:
    H["_freq_ranked_reference"] = [{"K": d["K"], "delta": d["delta_task_minus_recon"],
                                    "eval_n": d["eval_n"]} for d in _prev]
    print("\n  frequency-ranked (27f) vs random-K (here), delta task - recon:")
    _m = {}
    for d in RK: _m.setdefault(d["K"], []).append(d["delta_task_minus_recon"])
    for d in _prev:
        rk = _m.get(d["K"])
        print(f"      K={str(d['K']):>4}   freq-ranked {d['delta_task_minus_recon']:+.3f}"
              + (f"   random {sum(rk)/len(rk):+.3f}  (n={len(rk)} draws)" if rk else "   random n/a"))
RESULTS["factual_recall_unseen_randomK"] = H
print("\nEXP27h:", json.dumps(H, indent=2))


EXP27h  bin 570 = covered 486 + UNSEEN 84
      reusing arm A from EXP27f
      reusing arm B from EXP27f

    subset     n   native    recon  general     armA     armB
   covered   486    0.068    0.346    0.366    0.374    0.354
    UNSEEN    84    0.048    0.226    0.179    0.048    0.179
   fullbin   570    0.065    0.326    0.342    0.326    0.326

  PART 2: RANDOM-K selection (vs 27f's top-K-by-frequency), budget fixed at 387
     K  seed  classes  train n  ex/cls  eval n   recon    task   delta    shuf        p
      randK K=50 seed=0: 387 items, final mean CE 1.2558
    50     0       50      387     7.7      58   0.448   0.621  +0.172   0.000   0.0020
      randK K=50 seed=1: 387 items, final mean CE 0.5170
    50     1       46      387     8.4      54   0.315   0.519  +0.204   0.000   0.0034
      randK K=200 seed=0: 387 items, final mean CE 1.3588
   200     0      113      387     3.4     246   0.317   0.346  +0.028   0.000   0.1671
      randK K=200 seed=1: 387 items, fin

In [16]:
import json, datetime
_keys = [k for k in ("factual_recall_v2", "factual_recall_classcount",
                     "factual_recall_classmatch", "factual_recall_oracle_pooled") if k in RESULTS]
with open("exp27_followups_results.json", "w") as fh:
    json.dump({k: RESULTS[k] for k in _keys}, fh, indent=2)
print("wrote exp27_followups_results.json with:", _keys)
print("  at", datetime.datetime.now().isoformat(timespec="seconds"))
_missing = [k for k in ("factual_recall_classmatch", "factual_recall_oracle_pooled") if k not in RESULTS]
if _missing:
    print("  >>> WARNING: missing", _missing, "-- did those cells finish?")
print("\nNOTE: factual_recall_v2 here is a ONE-SEED rerun kept for provenance (it shows the bin")
print("      reproduced). Quote the 5-seed numbers from exp27_results.json, not these.")


wrote exp27_followups_results.json with: ['factual_recall_v2', 'factual_recall_classcount', 'factual_recall_classmatch', 'factual_recall_oracle_pooled']
  at 2026-09-11T19:46:10

NOTE: factual_recall_v2 here is a ONE-SEED rerun kept for provenance (it shows the bin
      reproduced). Quote the 5-seed numbers from exp27_results.json, not these.
